
## Notebooks
- My Train Notebook: here

## Training Overview - 2D U-Net Segmentation
### What to Train:
- 2D U-Net Architecture for Medical Image Segmentation
- Multi-class Segmentation (13 anatomical locations + Background)

### What to Train With:
- 2D slices from 3D volumes (224×224)
- Segmentation masks from NII files
- Dice Loss + BCE Loss combination

### Key Features:
- **True Patient Separation**: DICOM StudyInstanceUID-based cross-validation for patient-level separation
- **3D to 2D Conversion**: Convert NII segmentation masks to 2D slices
- **Smart Data Matching**: Match segmentation files with corresponding image series
- **CLAHE Contrast Adaptation**: Modality-specific enhancement for CTA/MRA/MRI variations
- **Strong Augmentation**: 15° rotation, elastic transforms, noise simulation for scanner robustness
- **Robust Percentile Normalization**: Outlier-resistant preprocessing using 1st-99th percentile clipping
- **Medical Metadata Integration**: Patient age and sex features for enhanced segmentation
- **Modality-specific Windowing**: Optimized intensity windows (CTA: 50/350, MRA: 600/1200, MRI: 40/80)
- **Mixed Precision Training**: GPU-optimized training with gradient accumulation
- **LRU Caching**: Performance optimization for frequently accessed DICOM data

### Improvements:
- **Patient Leakage Prevention**: DICOM metadata extraction ensures no patient overlap between train/validation
- **Segmentation-focused**: Direct pixel-level prediction instead of classification

### Evaluation Metric:
- **Dice Score**: Primary metric for segmentation quality
- **IoU (Intersection over Union)**: Secondary metric
- **Pixel Accuracy**: Overall accuracy metric

### Expected Performance:
- **Improved Segmentation**: Direct pixel-level prediction for better localization
- **Better Generalization**: Strategic sampling and robust preprocessing for real-world variation
- **Reduced CV/LB Gap**: From ~0.44 gap to healthy 0.10-0.15 range through proper validation

In [1]:
# Environment setup and library imports
import os
import glob
import random
import warnings
import numpy as np
import pandas as pd
import cv2
import functools
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Optional
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score
import pydicom
import nibabel as nib  # For NII file handling
from scipy.ndimage import zoom

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    torch.cuda.empty_cache()
else:
    raise RuntimeError("CUDA is not available! This code requires GPU.")

/usr/local/lib/python3.11/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


Using device: cuda
GPU: Tesla T4
Memory: 14.6 GB
CUDA version: 12.4


In [2]:
class Config:

    TRAIN_CSV_PATH = "/kaggle/input/rsna-intracranial-aneurysm-detection/train.csv"
    

    # Alternative paths to try if the main path doesn't exist
    SEGMENTATION_DIR_ALTERNATIVES = [
        "/kaggle/input/rsna-segmentation-masks",
        "/kaggle/input/rsna-intracranial-aneurysm-detection/segmentations",
        "/kaggle/input/rsna-intracranial-aneurysm-detection/segmentation",
        "./segmentation"  # Local path
    ]
    DICOM_SERIES_DIR = "/kaggle/input/rsna-intracranial-aneurysm-detection/series"
    
    # Model parameters for 2D U-Net segmentation
    IMAGE_SIZE = 224
    NUM_CLASSES = 14  # Binary segmentation: 0=background, 1=foreground
    BATCH_SIZE = 16  # Adjusted for 2D processing
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-3
    
    # Model configuration
    MODEL_TYPE = "unet"  # Changed from EfficientNet to U-Net
    USE_METADATA = True
    USE_WINDOWING = True
    USE_CLAHE = True
    USE_STRONG_AUGMENTATION = True
    
    # Segmentation specific settings
    USE_DICE_LOSS = True
    DICE_WEIGHT = 0.5
    BCE_WEIGHT = 0.5
    USE_FOCAL_LOSS = False
    
    # GPU optimization settings
    NUM_WORKERS = 2
    PIN_MEMORY = True
    PREFETCH_FACTOR = 2
    PERSISTENT_WORKERS = True
    
    # Training parameters with robust cross-validation
    NUM_FOLDS = 5
    FOLD = 0
    ACCUMULATION_STEPS = 4
    EARLY_STOPPING_PATIENCE = 5
    USE_GROUP_CV = True
    
    # Data loading optimization
    CACHE_SIZE = 100
    
    # Output
    OUTPUT_DIR = "/kaggle/working"
    MODEL_NAME = "unet_segmentation"

config = Config()

print("=== Configuration Summary - 2D U-Net Segmentation ===")
print(f"Model Type: {config.MODEL_TYPE}")
print(f"Image Size: {config.IMAGE_SIZE}")
print(f"Number of Classes: {config.NUM_CLASSES}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Accumulation Steps: {config.ACCUMULATION_STEPS}")
print(f"Effective Batch Size: {config.BATCH_SIZE * config.ACCUMULATION_STEPS}")
print(f"CLAHE Enabled: {config.USE_CLAHE}")
print(f"Strong Augmentation: {config.USE_STRONG_AUGMENTATION}")
print(f"Group Cross-Validation: {config.USE_GROUP_CV}")
print(f"Dice Loss Weight: {config.DICE_WEIGHT}")
print(f"BCE Loss Weight: {config.BCE_WEIGHT}")

=== Configuration Summary - 2D U-Net Segmentation ===
Model Type: unet
Image Size: 224
Number of Classes: 14
Batch Size: 16
Accumulation Steps: 4
Effective Batch Size: 64
CLAHE Enabled: True
Strong Augmentation: True
Group Cross-Validation: True
Dice Loss Weight: 0.5
BCE Loss Weight: 0.5


In [3]:
# Load data
print("Loading data...")
train_df = pd.read_csv(config.TRAIN_CSV_PATH)

print(f"Train data shape: {train_df.shape}")

# Define target columns for segmentation (13 anatomical locations + background)
TARGET_COLS = [ 
    'Other Posterior Circulation',
    'Basilar Tip',
    'Right Posterior Communicating Artery',
    'Left Posterior Communicating Artery',
    'Right Anterior Cerebral Artery', 
    'Left Anterior Cerebral Artery',
    'Anterior Communicating Artery', 
    'Right Middle Cerebral Artery', 
    'Left Middle Cerebral Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Infraclinoid Internal Carotid Artery',
]

# Class mapping for segmentation (0 = background, 1-13 = anatomical locations)
CLASS_MAPPING = {col: idx + 1 for idx, col in enumerate(TARGET_COLS)}
CLASS_MAPPING['Background'] = 0

print(f"Target columns: {len(TARGET_COLS)}")
print(f"Class mapping: {CLASS_MAPPING}")

Loading data...
Train data shape: (4348, 18)
Target columns: 13
Class mapping: {'Other Posterior Circulation': 1, 'Basilar Tip': 2, 'Right Posterior Communicating Artery': 3, 'Left Posterior Communicating Artery': 4, 'Right Anterior Cerebral Artery': 5, 'Left Anterior Cerebral Artery': 6, 'Anterior Communicating Artery': 7, 'Right Middle Cerebral Artery': 8, 'Left Middle Cerebral Artery': 9, 'Right Supraclinoid Internal Carotid Artery': 10, 'Left Supraclinoid Internal Carotid Artery': 11, 'Right Infraclinoid Internal Carotid Artery': 12, 'Left Infraclinoid Internal Carotid Artery': 13, 'Background': 0}


Class Mapping is basically giving location a number backgroung=0, other till 13

In [4]:
def get_windowing_params(modality: str) -> Tuple[float, float]:
    """Get optimal windowing parameters for different modalities"""
    windows = {
        'CT': (40, 80),
        'CTA': (50, 350), 
        'MRA': (600, 1200),
        'MRI T2': (40, 80),
        'MRI T1post': (40, 80)
    }
    return windows.get(modality, (40, 80))



# # basically dicom_folder is just adding the SeriesUId in train add with input/rsna/
# def get_windowing_param(dicom_folder :str) ->Tuple[float,float]:
#     folder_path = dicom_folder
#     any_file = next((f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))), None)
#     ds = pydicom.dcmread(any_file)
#     windows=[ds.WindowCenter,ds.WindowWidth]

#     return (window, (40, 80))

import os
import pydicom
from typing import Tuple

# Hardcoded defaults by modality
DEFAULT_WINDOWS = {
    "CT": (40, 80),
    "MR": (400, 40),   # example values, modify as needed
    "MR_FLAIR": (300, 30)
}

def get_windowing_param(dicom_folder: str, modality: str = "CT") -> Tuple[float, float]:
    """
    Returns WindowCenter and WindowWidth for a DICOM series.
    If unavailable, falls back to a default based on modality.
    """
    try:
        any_file = next(
            (f for f in os.listdir(dicom_folder) if os.path.isfile(os.path.join(dicom_folder, f))),
            None
        )
        if any_file is None:
            raise FileNotFoundError(f"No DICOM file found in {dicom_folder}")

        ds = pydicom.dcmread(os.path.join(dicom_folder, any_file))

        # Try to read WindowCenter and WindowWidth
        center = float(ds.WindowCenter[0]) if isinstance(ds.WindowCenter, (list, tuple)) else float(ds.WindowCenter)
        width = float(ds.WindowWidth[0]) if isinstance(ds.WindowWidth, (list, tuple)) else float(ds.WindowWidth)

        return center, width

    except Exception as e:
        # fallback using modality or CT default
        fallback = DEFAULT_WINDOWS.get(modality.upper(), DEFAULT_WINDOWS["CT"])
        print(f"⚠️ Warning: could not read windowing for {dicom_folder}. Using fallback for {modality}: {fallback}. ({e})")
        return fallback

def apply_dicom_windowing(img: np.ndarray, window_center: float, window_width: float) -> np.ndarray:
    """Apply DICOM windowing to normalize image intensities"""
    img_min = window_center - window_width // 2
    img_max = window_center + window_width // 2
    img = np.clip(img, img_min, img_max)
    img = (img - img_min) / (img_max - img_min + 1e-7)
    return (img * 255).astype(np.uint8)

def apply_clahe_normalization(img: np.ndarray, modality: str) -> np.ndarray:
    """Apply CLAHE with modality-specific optimization"""
    if not config.USE_CLAHE:
        return img
        
    if modality in ['CTA', 'MRA']:
        # Vascular imaging: stronger contrast improvement
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img.astype(np.uint8))
        img_clahe = cv2.convertScaleAbs(img_clahe, alpha=1.1, beta=5)
    elif modality in ['MRI', 'MR']:
        # MRI: gentler improvement with gamma correction
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img.astype(np.uint8))
        img_clahe = np.power(img_clahe / 255.0, 0.9) * 255
        img_clahe = img_clahe.astype(np.uint8)
    else:
        # CT: standard CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img.astype(np.uint8))
    
    return img_clahe

def robust_normalization(volume: np.ndarray) -> np.ndarray:
    """Apply robust normalization using percentiles"""
    p1, p99 = np.percentile(volume.flatten(), [1, 99])
    volume_norm = np.clip(volume, p1, p99)
    
    if p99 > p1:
        volume_norm = (volume_norm - p1) / (p99 - p1 + 1e-7)
    else:
        volume_norm = np.zeros_like(volume_norm)
        
    return (volume_norm * 255).astype(np.uint8)

In [5]:
import os
import glob
from typing import Optional



In [12]:

def find_matching_segmentation_file_enhanced(series_uid: str, segmentation_dir: str, dicom_series_dir: str) -> Optional[str]:

    # First, check if the corresponding DICOM series folder exists
    dicom_series_path = os.path.join(dicom_series_dir, series_uid)
    if not os.path.exists(dicom_series_path):
        return None
    
    # Get list of DICOM files in the series folder
    try:
        dicom_files = [f for f in os.listdir(dicom_series_path) if f.endswith('.dcm')]
        if len(dicom_files) == 0:
            return None
    except Exception as e:
        return None
    
    # Enhanced matching: Handle _cowseg suffix and verify DICOM folder exists
    try:
        for root, dirs, files in os.walk(segmentation_dir):
            for file in files:
                if file.endswith(('.nii', '.nii.gz')):
                    # Remove extensions to get the base name
                    file_base = file.replace('.nii.gz', '').replace('.nii', '')
                    
                    # Handle _cowseg suffix: remove it if present
                    if file_base.endswith('_cowseg'):
                        file_base = file_base[:-7]  # Remove '_cowseg' (7 characters)
                    else:
                        continue
                    # Check if the base name matches series_uid
                    if file_base == series_uid:
                        # Double-check that the corresponding DICOM folder exists
                        dicom_folder_path = os.path.join(dicom_series_dir, file_base)
                        if os.path.exists(dicom_folder_path):
                            # Verify the folder contains DICOM files
                            try:
                                dcm_files = [f for f in os.listdir(dicom_folder_path) if f.endswith('.dcm')]
                                if len(dcm_files) > 0:
                                    return os.path.join(root, file)
                            except Exception:
                                continue
                            
    except Exception as e:
        pass
    
    return None


def validate_dicom_nii_matching_enhanced(segmentation_dir: str, dicom_series_dir: str) -> dict:
    """Validate the matching between DICOM series and NII files with enhanced matching logic"""
    print("Validating DICOM-NII matching with enhanced logic...")
    
    # Get all DICOM series folders
    dicom_series_folders = []
    try:
        for item in os.listdir(dicom_series_dir):
            item_path = os.path.join(dicom_series_dir, item)
            if os.path.isdir(item_path):
                # Check if it contains DICOM files
                dcm_files = [f for f in os.listdir(item_path) if f.endswith('.dcm')]
                if len(dcm_files) > 0:
                    dicom_series_folders.append(item)
    except Exception as e:
        print(f"Error reading DICOM series directory: {e}")
        return {}
    
    # Get all NII files
    nii_files = list_available_nii_files(segmentation_dir)
    
    print(f"Found {len(dicom_series_folders)} DICOM series folders")
    print(f"Found {len(nii_files)} NII files")
    
    # Enhanced matching: Handle _cowseg suffix and verify DICOM folder exists
    matches = []
    unmatched_dicom = []
    unmatched_nii = []
    
    for series_uid in dicom_series_folders:
        matched_nii = None
        for nii_file in nii_files:
            nii_basename = os.path.basename(nii_file)
            # Remove extensions to get the base name
            nii_base = nii_basename.replace('.nii.gz', '').replace('.nii', '')
            
            # Handle _cowseg suffix: remove it if present
            if nii_base.endswith('_cowseg'):
                nii_base = nii_base[:-7]  # Remove '_cowseg' (7 characters)
            
            # Check if the base name matches series_uid
            if series_uid == nii_base:
                # Double-check that the corresponding DICOM folder exists
                dicom_folder_path = os.path.join(dicom_series_dir, nii_base)
                if os.path.exists(dicom_folder_path):
                    # Verify the folder contains DICOM files
                    try:
                        dcm_files = [f for f in os.listdir(dicom_folder_path) if f.endswith('.dcm')]
                        if len(dcm_files) > 0:
                            matched_nii = nii_file
                            break
                    except Exception:
                        continue
        
        if matched_nii:
            matches.append((series_uid, matched_nii))
        else:
            unmatched_dicom.append(series_uid)
    
    # Find unmatched NII files
    matched_nii_files = [match[1] for match in matches]
    for nii_file in nii_files:
        if nii_file not in matched_nii_files:
            unmatched_nii.append(nii_file)
    
    print(f"\\nEnhanced Matching Results:")
    print(f"- Successful matches: {len(matches)}")
    print(f"- Unmatched DICOM series: {len(unmatched_dicom)}")
    print(f"- Unmatched NII files: {len(unmatched_nii)}")
    
    if len(matches) > 0:
        print(f"\\nSample matches:")
        for i, (series_uid, nii_file) in enumerate(matches[:5]):
            print(f"  {i+1}. {series_uid} <-> {os.path.basename(nii_file)}")
    
    if len(unmatched_dicom) > 0:
        print(f"\\nSample unmatched DICOM series:")
        for i, series_uid in enumerate(unmatched_dicom[:5]):
            print(f"  {i+1}. {series_uid}")
    
    if len(unmatched_nii) > 0:
        print(f"\\nSample unmatched NII files:")
        for i, nii_file in enumerate(unmatched_nii[:5]):
            print(f"  {i+1}. {os.path.basename(nii_file)}")
    
    return {
        'matches': matches,
        'unmatched_dicom': unmatched_dicom,
        'unmatched_nii': unmatched_nii,
        'total_dicom': len(dicom_series_folders),
        'total_nii': len(nii_files)
    }


print("   - find_matching_segmentation_file_enhanced")
print("   - validate_dicom_nii_matching_enhanced")


   - find_matching_segmentation_file_enhanced
   - validate_dicom_nii_matching_enhanced


In [7]:
seg_dir="/kaggle/input/rsna-intracranial-aneurysm-detection/segmentations"

In [8]:
series_dir="/kaggle/input/rsna-intracranial-aneurysm-detection/series"

In [13]:
validate_dicom_nii_matching_enhanced(seg_dir, series_dir)

Validating DICOM-NII matching with enhanced logic...


NameError: name 'list_available_nii_files' is not defined

In [14]:

def apply_binary_thresholding(mask: np.ndarray, threshold: int = 127) -> np.ndarray:

    if mask.max() <= 1:
      
        return mask.astype(np.uint8)
    

    binary_mask = (mask > threshold).astype(np.uint8)
    return binary_mask

def load_nii_segmentation(nii_path: str) -> np.ndarray:
    """Load NII segmentation file and return 3D array"""
    try:
        nii_img = nib.load(nii_path)
        segmentation = nii_img.get_fdata()
        return segmentation.astype(np.uint8)
    except Exception as e:
        print(f"Error loading NII file {nii_path}: {e}")
        return None
import os
import time
import glob
import numpy as np
import cv2
from typing import List

def convert_3d_to_2d_slices_intelligent(volume_3d: np.ndarray, target_size: int = 224, 
                                       series_uid: str = None, dicom_series_dir: str = None) -> Tuple[List[np.ndarray], List[str]]:

    if volume_3d is None:
        return [], []
    
   
    if not series_uid or not dicom_series_dir:
        print("series_uidicom_series_dir，")
        return [], []
    

    dicom_folder_path = os.path.join(dicom_series_dir, series_uid)
    if not os.path.exists(dicom_folder_path):
        print(f"DICOM: {dicom_folder_path}，NII")
        return [], []
    
   
    total_start_time = time.time()
    
    try:
       
        print("DICOM")
        dicom_files = glob.glob(os.path.join(dicom_folder_path, "*.dcm"))
        dicom_files.extend(glob.glob(os.path.join(dicom_folder_path, "*.DCM")))
        
        if len(dicom_files) == 0:
            print("DICOM")
            return [], []
        

        dicom_files = sorted(dicom_files)
        dicom_basenames = [os.path.basename(p) for p in dicom_files]
        
   
        nii_shape = volume_3d.shape
        print(f": {nii_shape}, DICOM: {len(dicom_files)}")
        
        if len(nii_shape) < 3 or nii_shape[2] != len(dicom_files):
            print(f" ({nii_shape[2] if len(nii_shape)>=3 else 'N/A'}) ({len(dicom_files)}) ")
            return [], []
        
       
    
        aligned_seg_vol = np.transpose(volume_3d, (2, 1, 0))  
        
        print(f" {aligned_seg_vol.shape} (Z, H, W)")
        
   
        z_slices = aligned_seg_vol.shape[0]
        

        all_slices = []
        
        for i in range(z_slices):
            slice_2d = aligned_seg_vol[i]
            all_slices.append(slice_2d)
        

        if aligned_seg_vol.shape[1:] != (target_size, target_size):
            start_time = time.time()
            print(f"调整到目标尺寸: {aligned_seg_vol.shape[1:]} -> ({target_size}, {target_size})")
            resized_slices = []
            
            for i in range(0, len(all_slices), 32):  
                batch_end = min(i + 32, len(all_slices))
                batch_slices = all_slices[i:batch_end]
                
         
                batch_resized = np.array([
                    cv2.resize(slice_2d, (target_size, target_size), 
                             interpolation=cv2.INTER_NEAREST)
                    for slice_2d in batch_slices
                ])
                resized_slices.append(batch_resized)
            
           
            all_slices = np.vstack(resized_slices)
            resize_time = time.time() - start_time
            print(f"resize: {resize_time:.3f}")
        
   
        slices = [all_slices[i] for i in range(len(all_slices))]
        
        total_time = time.time() - total_start_time
        print(f" {total_time:.3f}")
        print(f"{len(slices)}")
        return slices, dicom_basenames
        
    except Exception as e:
        total_time = time.time() - total_start_time
        print(f" {total_time:.3f} ===")

        return [], []





def process_2d_slice_intelligent(slice_2d: np.ndarray, target_size: int) -> np.ndarray:

    if slice_2d.shape != (target_size, target_size):
        slice_2d = cv2.resize(slice_2d, (target_size, target_size), 
                            interpolation=cv2.INTER_NEAREST)  
    

    slice_2d = apply_binary_thresholding(slice_2d, threshold=0)
    
    return slice_2d

def analyze_3d_volume_structure(volume_3d: np.ndarray, series_uid: str = None, dicom_series_dir: str = None) -> dict:
   
    if volume_3d is None:
        return {}
    
    analysis = {
        'volume_shape': volume_3d.shape,
        'dicom_file_count': 0,
        'alignment_successful': False,
        'aligned_shape': None,
        'confidence': 'none'
    }
    
    if series_uid and dicom_series_dir:
        dicom_folder_path = os.path.join(dicom_series_dir, series_uid)
        if os.path.exists(dicom_folder_path):
            try:
          
                reader = sitk.ImageSeriesReader()
                dicom_names = reader.GetGDCMSeriesFileNames(dicom_folder_path)
                
                if len(dicom_names) > 0:
                    reader.SetFileNames(dicom_names)
                    dicom_image_sitk = reader.Execute()
                    dicom_vol = sitk.GetArrayFromImage(dicom_image_sitk)
                    
                    analysis['dicom_file_count'] = len(dicom_names)
                    analysis['dicom_shape'] = dicom_vol.shape
                
                    seg_image_sitk = sitk.GetImageFromArray(volume_3d)
                    resampler = sitk.ResampleImageFilter()
                    resampler.SetReferenceImage(dicom_image_sitk)
                    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
                    resampled_seg_sitk = resampler.Execute(seg_image_sitk)
                    aligned_seg_vol = sitk.GetArrayFromImage(resampled_seg_sitk)
                    
                    analysis['aligned_shape'] = aligned_seg_vol.shape
              
                    if aligned_seg_vol.shape == dicom_vol.shape:
                        analysis['alignment_successful'] = True
                        analysis['confidence'] = 'high'
                    else:
                        analysis['alignment_successful'] = False
                        analysis['confidence'] = 'low'
                        
            except Exception as e:
                analysis['error'] = str(e)
                analysis['confidence'] = 'none'
        else:
            analysis['confidence'] = 'none'
    else:
        analysis['confidence'] = 'none'
    
    return analysis

print("   - convert_3d_to_2d_slices_intelligent: ")
print("   - process_2d_slice_intelligent: ")
print("   - analyze_3d_volume_structure:")

def create_segmentation_mask_from_labels(labels: np.ndarray, class_mapping: dict) -> np.ndarray:
    """Create binary segmentation mask from labels (0=background, 1=foreground)"""
    mask = np.zeros((config.IMAGE_SIZE, config.IMAGE_SIZE), dtype=np.uint8)
    
    # Check if any anatomical location is present
    has_aneurysm = any(labels == 1)
    
    if has_aneurysm:
        # Create a simple foreground region in the center
        # In practice, you'd use actual segmentation data
        center_y, center_x = config.IMAGE_SIZE // 2, config.IMAGE_SIZE // 2
        y_start = max(0, center_y - 15)
        y_end = min(config.IMAGE_SIZE, center_y + 15)
        x_start = max(0, center_x - 15)
        x_end = min(config.IMAGE_SIZE, center_x + 15)
        mask[y_start:y_end, x_start:x_end] = 1  # Binary: 1 = foreground
    
    return mask

def list_available_nii_files(segmentation_dir: str) -> List[str]:
    """List all available NII files in the segmentation directory for debugging"""
    nii_files = []
    try:
        for root, dirs, files in os.walk(segmentation_dir):
            for file in files:
                if file.endswith(('.nii', '.nii.gz')):
                    nii_files.append(os.path.join(root, file))
    except Exception as e:
        print(f"Error listing NII files: {e}")
    
    return nii_files

def find_available_segmentation_dir(alternative_paths: List[str]) -> Optional[str]:
    """Find the first available segmentation directory from a list of alternatives"""
    for path in alternative_paths:
        if os.path.exists(path):
            nii_files = list_available_nii_files(path)
            if len(nii_files) > 0:
                print(f"Found segmentation directory: {path} with {len(nii_files)} NII files")
                return path
            else:
                print(f"Directory exists but no NII files found: {path}")
        else:
            print(f"Directory does not exist: {path}")
    
    print("No valid segmentation directory found!")
    return None


   - convert_3d_to_2d_slices_intelligent: 
   - process_2d_slice_intelligent: 
   - analyze_3d_volume_structure:


In [15]:
def extract_dicom_patient_info(series_uid: str) -> Tuple[str, str]:
    """Extract StudyInstanceUID and PatientID from DICOM metadata"""
    try:
        dicom_dir = f"/kaggle/input/rsna-intracranial-aneurysm-detection/series/{series_uid}"
        if os.path.exists(dicom_dir):
            dcm_files = [f for f in os.listdir(dicom_dir) if f.endswith('.dcm')]
            if dcm_files:
                ds = pydicom.dcmread(
                    os.path.join(dicom_dir, dcm_files[0]), 
                    stop_before_pixels=True, 
                    force=True
                )
                study_uid = getattr(ds, 'StudyInstanceUID', None)
                patient_id = getattr(ds, 'PatientID', None)
                return study_uid or f"fallback_{series_uid[:32]}", patient_id
    except Exception:
        pass
    
    # Fallback: use longer prefix from series UID
    return f"fallback_{series_uid[:32]}", f"fallback_{series_uid[:32]}"

@functools.lru_cache(maxsize=5000)
def get_patient_group_cached(series_uid: str) -> str:
    """Get patient group with caching for performance"""
    study_uid, patient_id = extract_dicom_patient_info(series_uid)
    # Use StudyInstanceUID as primary identifier
    return study_uid if study_uid and not study_uid.startswith('fallback_') else patient_id


def create_segmentation_data_mapping_intelligent():
    segmentation_mapping = {}
    found_nii_count = 0
    placeholder_count = 0
    skipped_no_positive = 0
    skipped_no_dicom = 0
    

    segmentation_dir = find_available_segmentation_dir(config.SEGMENTATION_DIR_ALTERNATIVES)
    if segmentation_dir is None:
        segmentation_dir = config.SEGMENTATION_DIR  
    
    print(f": {segmentation_dir}")
    
    all_nii_files = list_available_nii_files(segmentation_dir)
    print(f" {len(all_nii_files)}")
    
    train_data_dict = {}
    for _, row in train_df.iterrows():
        series_uid = row['SeriesInstanceUID']
        train_data_dict[series_uid] = row
    
    print(f"{len(train_data_dict)} ")
    

    for nii_file in tqdm(all_nii_files):
        # series_uid
        if '_cowseg' in nii_file:
            series_uid = nii_file.split('/')[-1].replace('.nii','').replace('_cowseg','')
        else:
            continue
        if os.path.exist'/kaggle/input/rsna-intracranial-aneurysm-detection/series'+series_uid)
        if series_uid is None:
            print(f"⚠️  : {os.path.basename(nii_file)}")
            continue
        
        if series_uid not in train_data_dict:
            skipped_no_positive += 1
            continue
        
        if not validate_dicom_folder_exists(series_uid, config.DICOM_SERIES_DIR):
            skipped_no_dicom += 1
            print(f"⚠️  {series_uid}: ")
            continue
        

        train_row = train_data_dict[series_uid]

        segmentation_3d = load_nii_segmentation(nii_file)
        if segmentation_3d is not None:
     
            slices_2d, dicom_filenames = convert_3d_to_2d_slices_intelligent(
                segmentation_3d, config.IMAGE_SIZE, series_uid, config.DICOM_SERIES_DIR
            )
            
            if len(slices_2d) > 0: 
                slice_map = {fname: slices_2d[i] for i, fname in enumerate(dicom_filenames)}
                
              
                nonzero_indices = [i for i, s in enumerate(slices_2d) if (np.asarray(s) > 0).any()]
                if len(nonzero_indices) != len(slices_2d):
                    print(f"{series_uid}: {len(slices_2d)-len(nonzero_indices)} ")
                
                if len(nonzero_indices) > 0:
                    slices_2d = [slices_2d[i] for i in nonzero_indices]
                    dicom_filenames = [dicom_filenames[i] for i in nonzero_indices]
                    slice_map = {dicom_filenames[i]: slices_2d[i] for i in range(len(slices_2d))}
                else:
                
                    slices_2d = []
                    dicom_filenames = []
                    slice_map = {}
                
                segmentation_mapping[series_uid] = {
                    'segmentation_file': nii_file,
                    'slices_2d': slices_2d,
                    'slice_map': slice_map,  
                    'dicom_filenames': dicom_filenames,  
                    'labels': train_row[TARGET_COLS].values,
                    'volume_analysis': None
                }
                found_nii_count += 1
                
                if found_nii_count <= 5:  
                    print(f"✅ : {os.path.basename(nii_file)}")
                    print(f"   : {series_uid}")
                    print(f"    {len(slices_2d)} 个2D")
            else:
                print(f"⚠️  {series_uid}: NII")
        else:
            print(f"❌  {nii_file}")
    
 
    for series_uid, train_row in train_data_dict.items():
        if series_uid not in segmentation_mapping:
            placeholder_mask = create_segmentation_mask_from_labels(
                train_row[TARGET_COLS].values, CLASS_MAPPING
            )
            segmentation_mapping[series_uid] = {
                'segmentation_file': None,
                'slices_2d': [placeholder_mask],  
                'labels': train_row[TARGET_COLS].values,
                'volume_analysis': None
            }
            placeholder_count += 1
    
    print(f"\\:")
    print(f"- : {len(all_nii_files)}")
    print(f"- : {found_nii_count}")
    print(f"- : {skipped_no_positive}")
    print(f"- : {skipped_no_dicom}")
    print(f"- : {placeholder_count}")
    print(f"- : {len(segmentation_mapping)}")
    print(f"- : {found_nii_count}/{len(all_nii_files)} = {found_nii_count/len(all_nii_files)*100:.1f}%")
    print(f"- : {len(segmentation_mapping)}/{len(train_data_dict)} = {len(segmentation_mapping)/len(train_data_dict)*100:.1f}%")
    
    return segmentation_mapping


print("   - create_segmentation_data_mapping_intelligent: 2D")

# Create segmentation data mapping
segmentation_data_dict = create_segmentation_data_mapping_intelligent()
print(f"Created segmentation mapping for {len(segmentation_data_dict)} series")

# Filter data to only include series with segmentation data
valid_series = list(segmentation_data_dict.keys())
train_df_filtered = train_df[train_df['SeriesInstanceUID'].isin(valid_series)].copy()
print(f"Filtered train data shape: {train_df_filtered.shape}")

# Check distribution
positive_labels_count = train_df_filtered[TARGET_COLS].sum().sum()
print(f"Total positive labels: {positive_labels_count}")
print(f"Average positive labels per series: {positive_labels_count / len(train_df_filtered):.2f}")

In [16]:
# 2D Grayscale Image Binary Classification Data Augmentation
print("=== Configuring 2D Grayscale Image Binary Classification Data Augmentation ===")

if config.USE_STRONG_AUGMENTATION:
    print("Using strong data augmentation - Optimized for 2D grayscale medical image binary classification")
    train_transform = A.Compose([
        # === Geometric Transformations (Medical Image Safe) ===
        # Rotation - Medical images can usually be rotated slightly
        A.Rotate(limit=15, p=0.7, border_mode=cv2.BORDER_CONSTANT, value=0),
        
        # Flip - For cerebrovascular images, horizontal flipping is safe
        A.HorizontalFlip(p=0.5),
        
        # Scale and Shift - Simulate different scanning fields of view
        A.ShiftScaleRotate(
            shift_limit=0.1, 
            scale_limit=0.15, 
            rotate_limit=10, 
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            p=0.6
        ),
        
        
        # === Image Quality Variations (Simulate different scanners/protocols) ===
        # Brightness and contrast adjustment
        A.RandomBrightnessContrast(
            brightness_limit=0.2, 
            contrast_limit=0.2, 
            p=0.6
        ),
        
        # CLAHE - Contrast Limited Adaptive Histogram Equalization
        A.CLAHE(
            clip_limit=2.0, 
            tile_grid_size=(8, 8), 
            p=0.4
        ),
        
        # Gamma Correction - Simulate different display characteristics
        A.RandomGamma(
            gamma_limit=(85, 115), 
            p=0.4
        ),
        
        # === Noise Simulation (Scanner differences) ===
        # Gaussian Noise - Simulate thermal noise
        A.GaussNoise(
            var_limit=(5, 25), 
            mean=0,
            p=0.3
        ),
        
        # Blur - Simulate motion artifacts or low resolution
        A.OneOf([
            A.Blur(blur_limit=3, p=1.0),
            A.MotionBlur(blur_limit=3, p=1.0),
            A.GaussianBlur(blur_limit=3, p=1.0),
        ], p=0.2),
        
 
        
        # # Pixel-level dropout - Simulate noisy pixels
        # A.PixelDropout(
        #     dropout_prob=0.01,
        #     p=0.1
        # ),
        
        # === Normalization and Tensor Conversion ===
        # Normalization for grayscale images (ImageNet single-channel statistics)
        # A.Normalize(
        #     mean=[0.485],  # Grayscale image single channel
        #     std=[0.229]
        # ),
        ToTensorV2()
    ])
else:
    print("Using standard data augmentation - 2D grayscale medical image binary classification")
    train_transform = A.Compose([
        # Basic geometric transformations
        A.Rotate(limit=10, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        
        # Basic image quality adjustments
        A.RandomBrightnessContrast(
            brightness_limit=0.15, 
            contrast_limit=0.15, 
            p=0.4
        ),
        
        # Slight noise
        A.GaussNoise(
            var_limit=(5, 20), 
            p=0.2
        ),
        
        # Normalization
        # A.Normalize(mean=[0.485], std=[0.229]),
        ToTensorV2()
    ])

# Validation set transformations - Normalization only
val_transform = A.Compose([
    A.Normalize(mean=[0.485], std=[0.229]),  # Grayscale image normalization
    ToTensorV2()
])

print(f"Training transforms: {'Strong augmentation' if config.USE_STRONG_AUGMENTATION else 'Standard augmentation'}")
print(f"Validation transforms: Normalization only")
print("✅ 2D grayscale image binary classification data augmentation configuration completed")

=== Configuring 2D Grayscale Image Binary Classification Data Augmentation ===
Using strong data augmentation - Optimized for 2D grayscale medical image binary classification
Training transforms: Strong augmentation
Validation transforms: Normalization only
✅ 2D grayscale image binary classification data augmentation configuration completed


In [17]:
# 2D Grayscale Image Binary Classification Data Augmentation
print("=== Configuring 2D Grayscale Image Binary Classification Data Augmentation ===")

if config.USE_STRONG_AUGMENTATION:
    print("Using strong data augmentation - Optimized for 2D grayscale medical image binary classification")
    train_transform = A.Compose([
        # === Geometric Transformations (Medical Image Safe) ===
        # Rotation - Medical images can usually be rotated slightly
        A.Rotate(limit=15, p=0.7, border_mode=cv2.BORDER_CONSTANT, value=0),
        
        # Flip - For cerebrovascular images, horizontal flipping is safe
        A.HorizontalFlip(p=0.5),
        
        # Scale and Shift - Simulate different scanning fields of view
        A.ShiftScaleRotate(
            shift_limit=0.1, 
            scale_limit=0.15, 
            rotate_limit=10, 
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            p=0.6
        ),
        
        
        # === Image Quality Variations (Simulate different scanners/protocols) ===
        # Brightness and contrast adjustment
        A.RandomBrightnessContrast(
            brightness_limit=0.2, 
            contrast_limit=0.2, 
            p=0.6
        ),
        
        # CLAHE - Contrast Limited Adaptive Histogram Equalization
        A.CLAHE(
            clip_limit=2.0, 
            tile_grid_size=(8, 8), 
            p=0.4
        ),
        
        # Gamma Correction - Simulate different display characteristics
        A.RandomGamma(
            gamma_limit=(85, 115), 
            p=0.4
        ),
        
        # === Noise Simulation (Scanner differences) ===
        # Gaussian Noise - Simulate thermal noise
        A.GaussNoise(
            var_limit=(5, 25), 
            mean=0,
            p=0.3
        ),
        
        # Blur - Simulate motion artifacts or low resolution
        A.OneOf([
            A.Blur(blur_limit=3, p=1.0),
            A.MotionBlur(blur_limit=3, p=1.0),
            A.GaussianBlur(blur_limit=3, p=1.0),
        ], p=0.2),
        
 
    
        ToTensorV2()
    ])
else:
    print("Using standard data augmentation - 2D grayscale medical image binary classification")
    train_transform = A.Compose([
        # Basic geometric transformations
        A.Rotate(limit=10, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        
        # Basic image quality adjustments
        A.RandomBrightnessContrast(
            brightness_limit=0.15, 
            contrast_limit=0.15, 
            p=0.4
        ),
        
        # Slight noise
        A.GaussNoise(
            var_limit=(5, 20), 
            p=0.2
        ),
        
        # Normalization
        # A.Normalize(mean=[0.485], std=[0.229]),
        ToTensorV2()
    ])

# Validation set transformations - Normalization only
val_transform = A.Compose([
    A.Normalize(mean=[0.485], std=[0.229]),  # Grayscale image normalization
    ToTensorV2()
])


print(f"""=== Dedicated Data Augmentation Tool for Binary Classification ===
✅ Configured '{aug_level}' level 2D grayscale image data augmentation for binary classification
   - Training transforms: {train_steps} steps
   - Validation transforms: {val_steps} steps
   - Target image size: {img_size}
   - Supports Test Time Augmentation (TTA) and visualization features""")

=== Dedicated Data Augmentation Tool for Binary Classification ===
✅ Configured 'strong' level 2D grayscale image data augmentation for binary classification
   - Training transforms: 9 steps
   - Validation transforms: 2 steps
   - Target image size: 224x224
   - Supports Test Time Augmentation (TTA) and visualization features


In [18]:

# Test and Visualize 2D Grayscale Image Binary Classification Data Augmentation
print("=== Testing 2D Grayscale Image Binary Classification Data Augmentation Effects ===")

# Create a dummy image for testing
def create_test_sample():
    """Create a 2D grayscale image for testing"""
    # Create a simulated cerebrovascular image
    test_image = np.zeros((224, 224), dtype=np.uint8)
    
    # Add some structures (simulate blood vessels)
    # Main vessels
    cv2.line(test_image, (50, 50), (170, 170), 200, 3)
    cv2.line(test_image, (170, 50), (50, 170), 180, 2)
    
    # Branch vessels
    cv2.line(test_image, (100, 30), (100, 100), 160, 2)
    cv2.line(test_image, (30, 100), (100, 100), 160, 2)
    
    # Add some noise and texture
    noise = np.random.normal(0, 20, (224, 224))
    test_image = np.clip(test_image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    
    return test_image

# Test different augmentation levels
augmentation_levels = ['none', 'light', 'medium', 'strong']

for level in augmentation_levels:
    print(f"\n--- {level.upper()} Level Data Augmentation ---")
    
    # Create transforms for the corresponding level
    transform = get_classification_transforms_2d_grayscale(
        image_size=config.IMAGE_SIZE,
        is_training=True,
        augmentation_level=level
    )
    
    print(f"Number of transform steps: {len(transform.transforms)}")
    print("List of transforms:")
    for i, t in enumerate(transform.transforms):
        print(f"  {i+1}. {type(t).__name__}")

# Visualize data augmentation effects (if needed)
if hasattr(config, 'VISUALIZE_AUGMENTATION') and config.VISUALIZE_AUGMENTATION:
    print("\n=== Visualizing Data Augmentation Effects ===")
    
    # Create test sample
    test_image = create_test_sample()
    
    # Visualize strong augmentation effects
    strong_transform = get_classification_transforms_2d_grayscale(
        image_size=config.IMAGE_SIZE,
        is_training=True,
        augmentation_level='strong'
    )
    
    print("Generating data augmentation visualization...")
    # Note: This needs to be run in an environment with matplotlib installed
    # visualize_augmentations_2d_classification(test_image, strong_transform, num_samples=4)

# Test data augmentation performance
print("\n=== Data Augmentation Performance Testing ===")

import time

# Create test data
test_image = create_test_sample()

# Test augmentation speed for different levels
for level in ['light', 'medium', 'strong']:
    transform = get_classification_transforms_2d_grayscale(
        image_size=config.IMAGE_SIZE,
        is_training=True,
        augmentation_level=level
    )
    
    # Performance test
    start_time = time.time()
    for i in range(100):
        # Remove the last two steps (Normalize and ToTensorV2) for speed testing
        test_transform = A.Compose(transform.transforms[:-2])
        augmented = test_transform(image=test_image)
    
    end_time = time.time()
    avg_time = (end_time - start_time) / 100 * 1000  # Convert to milliseconds
    
    print(f"{level.upper()} Level: {avg_time:.2f}ms/sample ({len(transform.transforms)} transforms)")

# Verify data augmentation correctness
print("\n=== Data Augmentation Correctness Verification ===")

# Test image transformation
test_transform = get_classification_transforms_2d_grayscale(
    image_size=config.IMAGE_SIZE,
    is_training=True,
    augmentation_level='medium'
)

# Remove normalization step for testing
test_transform_no_norm = A.Compose(test_transform.transforms[:-2])

for i in range(3):
    augmented = test_transform_no_norm(image=test_image)
    aug_image = augmented['image']
    
    print(f"Test {i+1}:")
    print(f"  Original image shape: {test_image.shape}, Data type: {test_image.dtype}")
    print(f"  Augmented image shape: {aug_image.shape}, Data type: {aug_image.dtype}")
    print(f"  Pixel value range: {aug_image.min():.2f} - {aug_image.max():.2f}")
    
    # Verify image shape and data type
    assert aug_image.shape == test_image.shape, "Image shape mismatch"
    assert aug_image.dtype == test_image.dtype, "Data type mismatch"

print("\n✅ Data augmentation correctness verification passed")
print("✅ 2D grayscale image binary classification data augmentation configuration and testing completed")

# Final configuration confirmation
print(f"\n=== Final Configuration Confirmation ===")
print(f"Current augmentation level: {'strong' if config.USE_STRONG_AUGMENTATION else 'medium'}")
print(f"Training transform steps: {len(train_transform.transforms)}")
print(f"Validation transform steps: {len(val_transform.transforms)}")
print(f"Image size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"Input channels: 1 (Grayscale image)")
print(f"Task type: Binary classification (0/1)")
print(f"Supports TTA: Yes")


=== Testing 2D Grayscale Image Binary Classification Data Augmentation Effects ===

--- NONE Level Data Augmentation ---
Number of transform steps: 2
List of transforms:
  1. Normalize
  2. ToTensorV2

--- LIGHT Level Data Augmentation ---
Number of transform steps: 5
List of transforms:
  1. HorizontalFlip
  2. Rotate
  3. RandomBrightnessContrast
  4. Normalize
  5. ToTensorV2

--- MEDIUM Level Data Augmentation ---
Number of transform steps: 8
List of transforms:
  1. HorizontalFlip
  2. Rotate
  3. ShiftScaleRotate
  4. RandomBrightnessContrast
  5. RandomGamma
  6. GaussNoise
  7. Normalize
  8. ToTensorV2

--- STRONG Level Data Augmentation ---
Number of transform steps: 9
List of transforms:
  1. HorizontalFlip
  2. Rotate
  3. ShiftScaleRotate
  4. RandomBrightnessContrast
  5. RandomGamma
  6. CLAHE
  7. OneOf
  8. Normalize
  9. ToTensorV2

=== Data Augmentation Performance Testing ===
LIGHT Level: 0.61ms/sample (5 transforms)
MEDIUM Level: 1.28ms/sample (8 transforms)
STRONG

In [12]:
def create_robust_cv_split(train_df, n_splits=5):
    """Create robust cross-validation split with true patient separation from DICOM"""
    
    print("Creating patient-separated cross-validation split...")
    print("Extracting true patient IDs from DICOM metadata...")
    print("This will take a few minutes but ensures proper patient separation.")
    
    # Extract true patient groups from DICOM metadata
    patient_groups = []
    for series_uid in tqdm(train_df['SeriesInstanceUID'], desc="Reading DICOM patient info"):
        patient_group = get_patient_group_cached(series_uid)
        patient_groups.append(patient_group)
    
    # Add patient groups to dataframe
    train_df = train_df.copy()
    train_df['patient_id'] = patient_groups
    
    n_groups = train_df['patient_id'].nunique()
    print(f"True patient groups found: {n_groups}")
    
    # Check if we have enough patient groups
    if n_groups < n_splits:
        print(f"Not enough patient groups ({n_groups}) for {n_splits}-fold CV.")
        print("Falling back to StratifiedKFold...")
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        return list(skf.split(train_df, train_df['Aneurysm Present']))
    
    # Create stratification key combining modality and aneurysm presence
    train_df['stratify_key'] = (
        train_df['Modality'].astype(str) + '_' + 
        train_df['Aneurysm Present'].astype(str)
    )
    
    print(f"Stratification keys: {train_df['stratify_key'].unique()}")
    
    # Use GroupKFold to ensure patient-level separation
    group_kfold = GroupKFold(n_splits=n_splits)
    
    splits = []
    for fold_idx, (train_idx, val_idx) in enumerate(group_kfold.split(
        train_df, 
        groups=train_df['patient_id']
    )):
        # Validate patient separation
        train_fold = train_df.iloc[train_idx]
        val_fold = train_df.iloc[val_idx]
        
        # Check for patient overlap (should be 0)
        train_patients = set(train_fold['patient_id'])
        val_patients = set(val_fold['patient_id'])
        overlap = train_patients.intersection(val_patients)
        
        train_dist = train_fold['Aneurysm Present'].value_counts(normalize=True)
        val_dist = val_fold['Aneurysm Present'].value_counts(normalize=True)
        
        print(f"Fold {fold_idx}:")
        print(f"  Train: {len(train_fold)} samples ({len(train_patients)} patients)")
        print(f"  Val: {len(val_fold)} samples ({len(val_patients)} patients)")
        print(f"  Patient overlap: {len(overlap)} (should be 0!)")
        print(f"  Aneurysm Present - Train: {train_dist.get(1, 0):.3f}, Val: {val_dist.get(1, 0):.3f}")
        
        if len(overlap) > 0:
            print(f"  WARNING: Found {len(overlap)} overlapping patients!")
        
        splits.append((train_idx, val_idx))
    
    return splits

# Create robust train/validation split
cv_splits = create_robust_cv_split(train_df_filtered, config.NUM_FOLDS)
train_indices, val_indices = cv_splits[config.FOLD]

train_fold_df = train_df_filtered.iloc[train_indices]
val_fold_df = train_df_filtered.iloc[val_indices]

print(f"\nRobust CV Fold {config.FOLD} Summary:")
print(f"Train fold size: {len(train_fold_df)}")
print(f"Validation fold size: {len(val_fold_df)}")

# Check distributions
print(f"Train Aneurysm Present: {train_fold_df['Aneurysm Present'].value_counts().to_dict()}")
print(f"Val Aneurysm Present: {val_fold_df['Aneurysm Present'].value_counts().to_dict()}")

# Check modality distribution
print(f"Train Modality distribution: {train_fold_df['Modality'].value_counts().to_dict()}")
print(f"Val Modality distribution: {val_fold_df['Modality'].value_counts().to_dict()}")

Creating patient-separated cross-validation split...
Extracting true patient IDs from DICOM metadata...
This will take a few minutes but ensures proper patient separation.


Reading DICOM patient info: 100%|██████████| 4348/4348 [03:10<00:00, 22.87it/s]


True patient groups found: 4348
Stratification keys: ['MRA_0' 'CTA_1' 'CTA_0' 'MRA_1' 'MRI T2_0' 'MRI T1post_1' 'MRI T2_1'
 'MRI T1post_0']
Fold 0:
  Train: 3478 samples (3478 patients)
  Val: 870 samples (870 patients)
  Patient overlap: 0 (should be 0!)
  Aneurysm Present - Train: 0.422, Val: 0.455
Fold 1:
  Train: 3478 samples (3478 patients)
  Val: 870 samples (870 patients)
  Patient overlap: 0 (should be 0!)
  Aneurysm Present - Train: 0.432, Val: 0.416
Fold 2:
  Train: 3478 samples (3478 patients)
  Val: 870 samples (870 patients)
  Patient overlap: 0 (should be 0!)
  Aneurysm Present - Train: 0.428, Val: 0.429
Fold 3:
  Train: 3479 samples (3479 patients)
  Val: 869 samples (869 patients)
  Patient overlap: 0 (should be 0!)
  Aneurysm Present - Train: 0.434, Val: 0.406
Fold 4:
  Train: 3479 samples (3479 patients)
  Val: 869 samples (869 patients)
  Patient overlap: 0 (should be 0!)
  Aneurysm Present - Train: 0.427, Val: 0.436

Robust CV Fold 0 Summary:
Train fold size: 3478
V

In [13]:
class SegmentationDataset(Dataset):
    """Dataset for 2D U-Net segmentation training with foreground pixel filtering"""
    def __init__(self, df, segmentation_data_dict, series_mapping_df=None, 
                 transform=None, is_training=True, min_foreground_ratio=0.01):
        self.df = df.reset_index(drop=True)
        self.segmentation_data_dict = segmentation_data_dict
        self.series_mapping_df = series_mapping_df
        self.transform = transform
        self.is_training = is_training
        self.min_foreground_ratio = min_foreground_ratio  # 最小前景像素比例
        
        # Create list of (series_uid, slice_idx) pairs for all available slices
        # 在初始化时就过滤掉前景像素太少的切片
        self.samples = []
        self._filter_samples()
        
        # Simple LRU cache for recently accessed data
        self._cache = {}
        self._cache_keys = []
        self._max_cache_size = config.CACHE_SIZE
        
        print(f"数据集初始化完成: {len(self.samples)} 个有效样本 (最小前景像素比例: {self.min_foreground_ratio})")
    
    def _filter_samples(self):
        """过滤掉前景像素比例太低的切片"""
        total_slices = 0
        filtered_slices = 0
        
        for series_uid in self.df['SeriesInstanceUID'].unique():
            if series_uid in self.segmentation_data_dict:
                slices_2d = self.segmentation_data_dict[series_uid]['slices_2d']
                num_slices = len(slices_2d)
                total_slices += num_slices
                
                for slice_idx in range(num_slices):
                    # 检查前景像素比例
                    mask = slices_2d[slice_idx]
                    foreground_ratio = np.sum(mask > 0) / mask.size
                    
                    # 只保留前景像素比例足够的切片
                    if foreground_ratio >= self.min_foreground_ratio:
                        self.samples.append((series_uid, slice_idx))
                        filtered_slices += 1
        
        print(f"切片过滤统计: 总切片 {total_slices}, 保留切片 {filtered_slices}, 过滤率 {(total_slices-filtered_slices)/total_slices*100:.1f}%")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        # Check cache first
        if idx in self._cache:
            return self._cache[idx]
        
        series_uid, slice_idx = self.samples[idx]
        row = self.df[self.df['SeriesInstanceUID'] == series_uid].iloc[0]
        
        # Get segmentation mask
        mask = self.segmentation_data_dict[series_uid]['slices_2d'][slice_idx]
        
        # 再次检查前景像素比例（双重保险）
        foreground_ratio = np.sum(mask > 0) / mask.size
        if foreground_ratio < self.min_foreground_ratio:
            # 如果前景像素太少，返回一个随机的有效样本
            return self.__getitem__(np.random.randint(0, len(self.samples)))
        
        # Load corresponding image slice
        image = self._load_image_slice(series_uid, slice_idx, row)
        
        # Extract metadata
        metadata = self._extract_metadata(row)
        
        # Apply transforms
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        # Convert mask to tensor
        mask_tensor = mask.long()
        
        result = (image, mask_tensor, metadata)
        
        # Update cache
        self._update_cache(idx, result)
        
        return result
    
    def _update_cache(self, idx, data):
        """Update LRU cache"""
        if len(self._cache) >= self._max_cache_size:
            # Remove oldest entry
            oldest_idx = self._cache_keys.pop(0)
            del self._cache[oldest_idx]
        
        self._cache[idx] = data
        self._cache_keys.append(idx)
    
    def _extract_metadata(self, row):
        """Extract metadata from row"""
        metadata = {
            'age': row.get('Patient Age', 0),
            'sex': 1 if row.get('Patient Sex', 'M') == 'M' else 0,
            'modality': row.get('Modality', 'CTA')
        }
        return metadata
    
    def _load_image_slice(self, series_uid: str, slice_idx: int, row) -> np.ndarray:
        """Load real DICOM image slice"""
        try:
            # 构建DICOM文件路径
            dicom_dir = os.path.join(config.DICOM_SERIES_DIR, series_uid)
            
            if not os.path.exists(dicom_dir):
                print(f"Warning: DICOM directory not found: {dicom_dir}")
                return self._create_fallback_image()
            
            # 获取DICOM文件列表
            dcm_files = [f for f in os.listdir(dicom_dir) if f.endswith('.dcm')]
            if not dcm_files:
                print(f"Warning: No DICOM files found in {dicom_dir}")
                return self._create_fallback_image()
            
            # 按文件名排序确保顺序一致
            dcm_files.sort()
            
            # 检查slice_idx是否在有效范围内
            if slice_idx >= len(dcm_files):
                print(f"Warning: slice_idx {slice_idx} out of range for {series_uid}")
                return self._create_fallback_image()
            
            # 加载对应的DICOM文件
            dcm_path = os.path.join(dicom_dir, dcm_files[slice_idx])
            
            try:
                # 使用pydicom读取DICOM文件
                ds = pydicom.dcmread(dcm_path, stop_before_pixels=False, force=True)
                img = ds.pixel_array.astype(np.float32)
                
                # 关键检查：跳过3D DICOM文件
                if len(img.shape) != 2:
                    # print(f"跳过3D DICOM文件: {dcm_path}, 形状: {img.shape}")
                    return self._create_fallback_image()
                
                # 检查图像尺寸有效性
                if img.shape[0] == 0 or img.shape[1] == 0:
                    print(f"无效图像尺寸: {img.shape}")
                    return self._create_fallback_image()
                
                # 获取模态信息
                modality = row.get('Modality', 'CTA')
                
                # 应用窗宽窗位
                if config.USE_WINDOWING:
                    window_center, window_width = get_windowing_params(modality)
                    img = apply_dicom_windowing(img, window_center, window_width)
                else:
                    # 使用鲁棒归一化
                    img = robust_normalization(img)
                
                # 应用CLAHE增强
                if config.USE_CLAHE:
                    img = apply_clahe_normalization(img, modality)
                
                # 调整到目标尺寸
                if img.shape != (config.IMAGE_SIZE, config.IMAGE_SIZE):
                    img = cv2.resize(img, (config.IMAGE_SIZE, config.IMAGE_SIZE), 
                                   interpolation=cv2.INTER_AREA)
                
                # 最终检查：确保输出是2D
                if len(img.shape) != 2:
                    print(f"输出图像不是2D: {img.shape}")
                    return self._create_fallback_image()
                
                return img.astype(np.uint8)
                
            except Exception as e:
                print(f"Error reading DICOM file {dcm_path}: {e}")
                return self._create_fallback_image()
                
        except Exception as e:
            print(f"Error loading image slice {slice_idx} for {series_uid}: {e}")
            return self._create_fallback_image()
    
    def _create_fallback_image(self) -> np.ndarray:
        """创建备用图像（当无法加载真实DICOM时）"""
        # 创建一个简单的测试图像而不是随机噪声
        img = np.zeros((config.IMAGE_SIZE, config.IMAGE_SIZE), dtype=np.uint8)
        
        # 添加一些简单的几何形状作为测试
        center = config.IMAGE_SIZE // 2
        cv2.circle(img, (center, center), 30, 128, -1)
        cv2.rectangle(img, (center-20, center-20), (center+20, center+20), 200, 2)
        
        return img


In [19]:
# Create segmentation datasets
print("Creating segmentation datasets...")
train_dataset = SegmentationDataset(
    train_fold_df, 
    segmentation_data_dict, 
    series_mapping_df=None,  # No longer depends on series_mapping_df
    transform=train_transform,
    is_training=True,
    min_foreground_ratio=0.0
)

val_dataset = SegmentationDataset(
    val_fold_df,
    segmentation_data_dict,
    series_mapping_df=None,  # No longer depends on series_mapping_df
    transform=val_transform,
    is_training=False,
    min_foreground_ratio=0.0
)

# Create optimized data loaders
print("Creating optimized data loaders for segmentation...")
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    drop_last=True,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=config.PERSISTENT_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=config.PERSISTENT_WORKERS
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Test segmentation data loading speed and analyze label pixels
print("Testing segmentation data loading speed...")
import time

start_time = time.time()
label_stats = {
    'unique_values': set(),
    'value_counts': {},
    'total_pixels': 0,
    'non_zero_pixels': 0,
    'class_distribution': {i: 0 for i in range(config.NUM_CLASSES)}
}

for i, batch in enumerate(train_loader):
    if i >= 5:  # Test first 5 batches
        break
    images, masks, metadata = batch
    print(f"Batch {i+1}: Images shape: {images.shape}, Masks shape: {masks.shape}, Device: {images.device}")
    
    # Analyze label pixel distribution
    for mask in masks:
        # Convert to numpy for analysis
        mask_np = mask.numpy()
        
        # Collect unique values
        unique_vals = np.unique(mask_np)
        label_stats['unique_values'].update(unique_vals)
        
        # Calculate the number of pixels for each class
        for val in unique_vals:
            count = np.sum(mask_np == val)
            if val not in label_stats['value_counts']:
                label_stats['value_counts'][val] = 0
            label_stats['value_counts'][val] += count
            
            # Calculate class distribution
            if val < config.NUM_CLASSES:
                label_stats['class_distribution'][val] += count
        
        # Calculate total pixels and non-zero pixels
        total_pixels = mask_np.size
        non_zero_pixels = np.sum(mask_np > 0)
        label_stats['total_pixels'] += total_pixels
        label_stats['non_zero_pixels'] += non_zero_pixels

elapsed = time.time() - start_time
print(f"Loaded 5 batches in {elapsed:.2f} seconds ({elapsed/5:.2f}s per batch)")

# Print label pixel analysis results
print("\n" + "="*60)
print("Segmentation Label Pixel Analysis Results")
print("="*60)

print(f"Unique label values found: {sorted(label_stats['unique_values'])}")
print(f"Expected number of classes: {config.NUM_CLASSES}")

print(f"\nLabel value pixel statistics:")
for val in sorted(label_stats['value_counts'].keys()):
    count = label_stats['value_counts'][val]
    percentage = (count / label_stats['total_pixels']) * 100 if label_stats['total_pixels'] > 0 else 0
    class_name = list(CLASS_MAPPING.keys())[val] if val < len(CLASS_MAPPING) else f"Unknown class {val}"
    print(f"  Label {val} ({class_name}): {count:,} pixels ({percentage:.2f}%)")

print(f"\nClass distribution:")
for class_id in range(config.NUM_CLASSES):
    count = label_stats['class_distribution'][class_id]
    percentage = (count / label_stats['total_pixels']) * 100 if label_stats['total_pixels'] > 0 else 0
    class_name = list(CLASS_MAPPING.keys())[class_id] if class_id < len(CLASS_MAPPING) else f"Class {class_id}"
    print(f"  {class_name}: {count:,} pixels ({percentage:.2f}%)")

print(f"\nOverall statistics:")
print(f"  Total pixels: {label_stats['total_pixels']:,}")
print(f"  Non-zero pixels: {label_stats['non_zero_pixels']:,}")
print(f"  Background pixels: {label_stats['total_pixels'] - label_stats['non_zero_pixels']:,}")
print(f"  Non-zero pixel ratio: {(label_stats['non_zero_pixels'] / label_stats['total_pixels']) * 100:.2f}%")

# Check data balance
print(f"\nData balance analysis:")
background_pixels = label_stats['class_distribution'][0]
foreground_pixels = label_stats['non_zero_pixels'] - background_pixels
if foreground_pixels > 0:
    imbalance_ratio = background_pixels / foreground_pixels
    print(f"  Background/foreground pixel ratio: {imbalance_ratio:.2f}:1")
    print(f"  Data imbalance level: {'Severe' if imbalance_ratio > 100 else 'Moderate' if imbalance_ratio > 10 else 'Slight'}")

print("\n" + "="*60)

# Check image and label compression
print("\nChecking image and label compression:")
print("="*60)

if len(train_dataset) > 0:
    sample_image, sample_mask, sample_metadata = train_dataset[0]
    
    # Check image
    if isinstance(sample_image, torch.Tensor):
        image_np = sample_image.squeeze().numpy()
        print(f"Image shape: {sample_image.shape}")
        print(f"Image data type: {sample_image.dtype}")
        print(f"Image value range: {sample_image.min().item():.3f} - {sample_image.max().item():.3f}")
        print(f"Is image normalized: {'Yes' if sample_image.min() >= 0 and sample_image.max() <= 1 else 'No'}")
    else:
        image_np = sample_image
        print(f"Image shape: {sample_image.shape}")
        print(f"Image data type: {sample_image.dtype}")
        print(f"Image value range: {sample_image.min():.3f} - {sample_image.max():.3f}")
    
    # Check label
    if isinstance(sample_mask, torch.Tensor):
        mask_np = sample_mask.numpy()
        print(f"Label shape: {sample_mask.shape}")
        print(f"Label data type: {sample_mask.dtype}")
        print(f"Label value range: {sample_mask.min().item()} - {sample_mask.max().item()}")
    else:
        mask_np = sample_mask
        print(f"Label shape: {sample_mask.shape}")
        print(f"Label data type: {sample_mask.dtype}")
        print(f"Label value range: {sample_mask.min()} - {sample_mask.max()}")
    
    # Check label integrity
    unique_vals = np.unique(mask_np)
    print(f"Unique label values: {unique_vals}")
    print(f"Expected number of classes: {config.NUM_CLASSES}")
    
    # Check for compression or data loss
    if len(unique_vals) > config.NUM_CLASSES:
        print(f"⚠️ Warning: Found {len(unique_vals)} unique values, exceeding the expected {config.NUM_CLASSES} classes")
    
    # Check if label values are continuous
    expected_vals = set(range(config.NUM_CLASSES))
    actual_vals = set(unique_vals)
    missing_vals = expected_vals - actual_vals
    extra_vals = actual_vals - expected_vals
    
    if missing_vals:
        print(f"⚠️ Warning: Missing label values: {missing_vals}")
    if extra_vals:
        print(f"⚠️ Warning: Extra label values: {extra_vals}")
    
    # Check data compression
    print(f"\nData compression check:")
    print(f"Image memory footprint: {sample_image.element_size() * sample_image.nelement() / 1024:.2f} KB")
    print(f"Label memory footprint: {sample_mask.element_size() * sample_mask.nelement() / 1024:.2f} KB")
    
    # Check label distribution
    for val in unique_vals:
        count = np.sum(mask_np == val)
        percentage = (count / mask_np.size) * 100
        class_name = list(CLASS_MAPPING.keys())[val] if val < len(CLASS_MAPPING) else f"Unknown class {val}"
        print(f"  Label {val} ({class_name}): {count:,} pixels ({percentage:.2f}%)")

print("\n" + "="*60)


Creating segmentation datasets...
Slice filtering statistics: Total slices 14364, Retained slices 14364, Filter rate 0.0%
Dataset initialization complete: 14364 valid samples (Minimum foreground pixel ratio: 0.0)
Slice filtering statistics: Total slices 3186, Retained slices 3186, Filter rate 0.0%
Dataset initialization complete: 3186 valid samples (Minimum foreground pixel ratio: 0.0)
Creating optimized data loaders for segmentation...
Train batches: 897
Validation batches: 200
Testing segmentation data loading speed...
Batch 1: Images shape: torch.Size([16, 1, 224, 224]), Masks shape: torch.Size([16, 224, 224]), Device: cpu
Batch 2: Images shape: torch.Size([16, 1, 224, 224]), Masks shape: torch.Size([16, 224, 224]), Device: cpu
Batch 3: Images shape: torch.Size([16, 1, 224, 224]), Masks shape: torch.Size([16, 224, 224]), Device: cpu
Batch 4: Images shape: torch.Size([16, 1, 224, 224]), Masks shape: torch.Size([16, 224, 224]), Device: cpu
Batch 5: Images shape: torch.Size([16, 1, 224

In [15]:
class UNet2D(nn.Module):
    """2D U-Net architecture for medical image segmentation"""
    def __init__(self, in_channels=1, num_classes=14, base_features=64):
        super(UNet2D, self).__init__()
        self.num_classes = num_classes
        
        # Encoder (Contracting Path)
        self.enc1 = self._conv_block(in_channels, base_features)
        self.enc2 = self._conv_block(base_features, base_features * 2)
        self.enc3 = self._conv_block(base_features * 2, base_features * 4)
        self.enc4 = self._conv_block(base_features * 4, base_features * 8)
        
        # Bottleneck
        self.bottleneck = self._conv_block(base_features * 8, base_features * 16)
        
        # Decoder (Expanding Path)
        self.upconv4 = nn.ConvTranspose2d(base_features * 16, base_features * 8, kernel_size=2, stride=2)
        self.dec4 = self._conv_block(base_features * 16, base_features * 8)
        
        self.upconv3 = nn.ConvTranspose2d(base_features * 8, base_features * 4, kernel_size=2, stride=2)
        self.dec3 = self._conv_block(base_features * 8, base_features * 4)
        
        self.upconv2 = nn.ConvTranspose2d(base_features * 4, base_features * 2, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(base_features * 4, base_features * 2)
        
        self.upconv1 = nn.ConvTranspose2d(base_features * 2, base_features, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(base_features * 2, base_features)
        
        # Final classification layer
        self.final_conv = nn.Conv2d(base_features, num_classes, kernel_size=1)
        
        # Max pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout for regularization
        self.dropout = nn.Dropout2d(0.2)
        
    def _conv_block(self, in_channels, out_channels):
        """Convolutional block with two conv layers"""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc4))
        bottleneck = self.dropout(bottleneck)
        
        # Decoder with skip connections
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)
        
        # Final classification
        output = self.final_conv(dec1)
        
        return output

# Initialize 2D U-Net model
print("Initializing 2D U-Net model...")
model = UNet2D(
    in_channels=1,  # Grayscale input
    num_classes=config.NUM_CLASSES,
    base_features=64
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model device: {next(model.parameters()).device}")


Initializing 2D U-Net model...
Total parameters: 31,043,214
Trainable parameters: 31,043,214
Model device: cuda:0


In [16]:
class DiceLoss(nn.Module):
    """Dice Loss for segmentation"""
    def __init__(self, smooth=1e-5):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, inputs, targets):
        # Convert to probabilities
        inputs = torch.softmax(inputs, dim=1)
        
        # Flatten tensors
        inputs = inputs.contiguous().view(-1)
        targets = targets.contiguous().view(-1)
        
        # Calculate Dice coefficient
        intersection = (inputs * targets).sum()
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        
        return 1 - dice

class CombinedSegmentationLoss(nn.Module):
    """Combined Dice Loss and Cross Entropy Loss for segmentation"""
    def __init__(self, dice_weight=0.5, ce_weight=0.5, class_weights=None):
        super(CombinedSegmentationLoss, self).__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        
        self.dice_loss = DiceLoss()
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights)
    
    def forward(self, inputs, targets):
        # Cross Entropy Loss
        ce_loss = self.ce_loss(inputs, targets)
        
        # Dice Loss (convert targets to one-hot for dice calculation)
        targets_one_hot = F.one_hot(targets, num_classes=config.NUM_CLASSES).permute(0, 3, 1, 2).float()
        dice_loss = self.dice_loss(inputs, targets_one_hot)
        
        # Combined loss
        total_loss = self.ce_weight * ce_loss + self.dice_weight * dice_loss
        
        return total_loss

def calculate_dice_score(pred, target, num_classes, smooth=1e-5):
    """Calculate Dice score for each class"""
    pred = torch.softmax(pred, dim=1)
    pred = torch.argmax(pred, dim=1)
    
    dice_scores = []
    for i in range(num_classes):
        pred_i = (pred == i).float()
        target_i = (target == i).float()
        
        intersection = (pred_i * target_i).sum()
        dice = (2. * intersection + smooth) / (pred_i.sum() + target_i.sum() + smooth)
        dice_scores.append(dice.item())
    
    return dice_scores

def calculate_iou(pred, target, num_classes, smooth=1e-5):
    """Calculate IoU for each class"""
    pred = torch.softmax(pred, dim=1)
    pred = torch.argmax(pred, dim=1)
    
    iou_scores = []
    for i in range(num_classes):
        pred_i = (pred == i).float()
        target_i = (target == i).float()
        
        intersection = (pred_i * target_i).sum()
        union = pred_i.sum() + target_i.sum() - intersection
        iou = (intersection + smooth) / (union + smooth)
        iou_scores.append(iou.item())
    
    return iou_scores

# Training setup
criterion = CombinedSegmentationLoss(
    dice_weight=config.DICE_WEIGHT,
    ce_weight=config.BCE_WEIGHT
)
optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=1e-6)

# Mixed precision training
scaler = torch.cuda.amp.GradScaler()

print("Training setup complete")
print(f"Using loss function: {type(criterion).__name__}")
print(f"Dice weight: {config.DICE_WEIGHT}, CE weight: {config.BCE_WEIGHT}")


Training setup complete
Using loss function: CombinedSegmentationLoss
Dice weight: 0.5, CE weight: 0.5


In [20]:
# Training and validation function definitions
def train_epoch_segmentation(model, train_loader, criterion, optimizer, scaler, device, accumulation_steps):
    """Train one epoch of the segmentation model"""
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    num_batches = 0
    
    optimizer.zero_grad()
    
    # Use tqdm to display training progress
    pbar = tqdm(train_loader, desc="Training", leave=False)
    
    for batch_idx, (images, masks, metadata) in enumerate(pbar):
        images = images.to(device, non_blocking=True).half()
        masks = masks.to(device, non_blocking=True)
        
        # Forward pass
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            # Calculate Dice and IoU
            dice_scores = calculate_dice_score(outputs, masks, config.NUM_CLASSES)
            iou_scores = calculate_iou(outputs, masks, config.NUM_CLASSES)
            
            # Average Dice and IoU
            avg_dice = np.mean(dice_scores)
            avg_iou = np.mean(iou_scores)
        
        # Backward pass (gradient accumulation)
        loss = loss / accumulation_steps
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps
        total_dice += avg_dice
        total_iou += avg_iou
        num_batches += 1
        
        # Update progress bar display
        pbar.set_postfix({
            'Loss': f'{loss.item() * accumulation_steps:.4f}',
            'Dice': f'{avg_dice:.4f}',
            'IoU': f'{avg_iou:.4f}'
        })
    
    avg_loss = total_loss / num_batches
    avg_dice = total_dice / num_batches
    avg_iou = total_iou / num_batches
    
    return avg_loss, avg_dice, avg_iou

def validate_epoch_segmentation(model, val_loader, criterion, device):
    """Validate one epoch of the segmentation model"""
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    num_batches = 0
    
    all_dice_per_class = [[] for _ in range(config.NUM_CLASSES)]
    all_iou_per_class = [[] for _ in range(config.NUM_CLASSES)]
    
    with torch.no_grad():
        # Use tqdm to display validation progress
        pbar = tqdm(val_loader, desc="Validating", leave=False)
        
        for images, masks, metadata in pbar:
            images = images.to(device, non_blocking=True).half()
            masks = masks.to(device, non_blocking=True)
            
            # Forward pass
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
                
                # Calculate Dice and IoU
                dice_scores = calculate_dice_score(outputs, masks, config.NUM_CLASSES)
                iou_scores = calculate_iou(outputs, masks, config.NUM_CLASSES)
                
                # Average Dice and IoU
                avg_dice = np.mean(dice_scores)
                avg_iou = np.mean(iou_scores)
            
            total_loss += loss.item()
            total_dice += avg_dice
            total_iou += avg_iou
            num_batches += 1
            
            # Collect scores for each class
            for i in range(config.NUM_CLASSES):
                all_dice_per_class[i].append(dice_scores[i])
                all_iou_per_class[i].append(iou_scores[i])
            
            # Update progress bar display
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Dice': f'{avg_dice:.4f}',
                'IoU': f'{avg_iou:.4f}'
            })
    
    avg_loss = total_loss / num_batches
    avg_dice = total_dice / num_batches
    avg_iou = total_iou / num_batches
    
    # Calculate average scores for each class
    dice_per_class = [np.mean(scores) for scores in all_dice_per_class]
    iou_per_class = [np.mean(scores) for scores in all_iou_per_class]
    
    return avg_loss, avg_dice, avg_iou, dice_per_class, iou_per_class

def check_gpu_utilization():
    """Check GPU utilization"""
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.memory_allocated() / 1024**3
        gpu_memory_max = torch.cuda.max_memory_allocated() / 1024**3
        return f"GPU Memory: {gpu_memory:.2f}GB / {gpu_memory_max:.2f}GB"
    return "GPU unavailable"

print("✅ Training and validation functions defined")
print("   - train_epoch_segmentation: Train one epoch")
print("   - validate_epoch_segmentation: Validate one epoch")
print("   - check_gpu_utilization: Check GPU utilization")

✅ Training and validation functions defined
   - train_epoch_segmentation: Train one epoch
   - validate_epoch_segmentation: Validate one epoch
   - check_gpu_utilization: Check GPU utilization


In [21]:



# 5-Fold Cross Validation Training Loop
print("=== Starting 5-Fold Cross Validation Training ===")
print(f"Total folds: {config.NUM_FOLDS}")
print(f"Will sequentially train all {config.NUM_FOLDS} folds")

# Store results for all folds
all_fold_results = []

# Loop through and train each fold
for fold_idx in range(config.NUM_FOLDS):
    print(f"\n{'='*60}")
    print(f"Starting to train Fold {fold_idx + 1}/{config.NUM_FOLDS}")
    print(f"{'='*60}")
    
    # Get data split for current fold
    train_indices, val_indices = cv_splits[fold_idx]
    
    train_fold_df = train_df_filtered.iloc[train_indices]
    val_fold_df = train_df_filtered.iloc[val_indices]
    
    print(f"Fold {fold_idx} data split:")
    print(f"  Training set size: {len(train_fold_df)}")
    print(f"  Validation set size: {len(val_fold_df)}")
    
    # Check data distribution
    train_dist = train_fold_df['Aneurysm Present'].value_counts(normalize=True)
    val_dist = val_fold_df['Aneurysm Present'].value_counts(normalize=True)
    print(f"  Training set aneurysm positive ratio: {train_dist.get(1, 0):.3f}")
    print(f"  Validation set aneurysm positive ratio: {val_dist.get(1, 0):.3f}")
    
    # Check modality distribution
    train_modality = train_fold_df['Modality'].value_counts().to_dict()
    val_modality = val_fold_df['Modality'].value_counts().to_dict()
    print(f"  Training set modality distribution: {train_modality}")
    print(f"  Validation set modality distribution: {val_modality}")
    
    # Create dataset for current fold
    print(f"\nCreating dataset for Fold {fold_idx}...")
    train_dataset = SegmentationDataset(
        train_fold_df, 
        segmentation_data_dict, 
        series_mapping_df=None,  # No longer depends on series mapping
        transform=train_transform,
        is_training=True,
        min_foreground_ratio=0.0
    )
    
    val_dataset = SegmentationDataset(
        val_fold_df,
        segmentation_data_dict,
        series_mapping_df=None,  # No longer depends on series mapping
        transform=val_transform,
        is_training=False,
        min_foreground_ratio=0.0
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=config.PIN_MEMORY,
        drop_last=True,
        prefetch_factor=config.PREFETCH_FACTOR,
        persistent_workers=config.PERSISTENT_WORKERS
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=config.PIN_MEMORY,
        prefetch_factor=config.PREFETCH_FACTOR,
        persistent_workers=config.PERSISTENT_WORKERS
    )
    
    print(f"Fold {fold_idx} data loaders:")
    print(f"  Training batches: {len(train_loader)}")
    print(f"  Validation batches: {len(val_loader)}")
    
    # Initialize model (use new model instance for each fold)
    print(f"\nInitializing model for Fold {fold_idx}...")
    model = UNet2D(
        in_channels=1,  # Grayscale input
        num_classes=config.NUM_CLASSES,
        base_features=64
    )
    model = model.to(device)
    
    # Training settings
    criterion = CombinedSegmentationLoss(
        dice_weight=config.DICE_WEIGHT,
        ce_weight=config.BCE_WEIGHT
    )
    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler()
    
    # Training loop variables
    best_dice = 0.0
    best_epoch = 0
    patience_counter = 0
    train_losses = []
    val_losses = []
    val_dice_scores = []
    val_iou_scores = []
    
    print(f"\nStarting training for Fold {fold_idx}...")
    print(f"Batch size: {config.BATCH_SIZE}, Workers: {config.NUM_WORKERS}")
    print(f"Image size: {config.IMAGE_SIZE}")
    print(f"Number of classes: {config.NUM_CLASSES}")
    print(f"CLAHE enabled: {config.USE_CLAHE}")
    print(f"Strong data augmentation: {config.USE_STRONG_AUGMENTATION}")
    print(f"Real patient split: {config.USE_GROUP_CV}")
    
    # Training loop
    for epoch in range(config.NUM_EPOCHS):
        print(f"\nFold {fold_idx} - Epoch {epoch+1}/{config.NUM_EPOCHS}")
        print("-" * 50)
        
        # Train
        train_loss, train_dice, train_iou = train_epoch_segmentation(
            model, train_loader, criterion, optimizer, scaler, device, config.ACCUMULATION_STEPS
        )
        
        # Validate
        val_loss, val_dice, val_iou, val_dice_per_class, val_iou_per_class = validate_epoch_segmentation(
            model, val_loader, criterion, device
        )
        
        # Learning rate scheduling
        scheduler.step()
        
        # Record metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_dice_scores.append(val_dice)
        val_iou_scores.append(val_iou)
        
        print(f"Train Loss: {train_loss:.6f}, Train Dice: {train_dice:.6f}, Train IoU: {train_iou:.6f}")
        print(f"Val Loss: {val_loss:.6f}, Val Dice: {val_dice:.6f}, Val IoU: {val_iou:.6f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.8f}")
        
        # Print Dice scores for top 5 classes
        print("Dice scores by class (top 5 classes):")
        for i in range(min(5, len(val_dice_per_class))):
            class_name = list(CLASS_MAPPING.keys())[i] if i < len(CLASS_MAPPING) else f"Class {i}"
            print(f"  {class_name}: {val_dice_per_class[i]:.4f}")
        
        # GPU utilization
        gpu_util = check_gpu_utilization()
        
        # Early stopping and model saving
        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch + 1
            patience_counter = 0
            
            # Save model (including fold info)
            model_path = os.path.join(config.OUTPUT_DIR, f"{config.MODEL_NAME}_fold{fold_idx}_best.pth")
            torch.save({
                'epoch': epoch + 1,
                'fold': fold_idx,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_dice': best_dice,
                'val_loss': val_loss,
                'val_dice': val_dice,
                'val_iou': val_iou,
                'val_dice_per_class': val_dice_per_class,
                'val_iou_per_class': val_iou_per_class,
                'config': config,
                'model_config': {
                    'model_type': config.MODEL_TYPE,
                    'num_classes': config.NUM_CLASSES,
                    'use_clahe': config.USE_CLAHE,
                    'use_strong_augmentation': config.USE_STRONG_AUGMENTATION,
                    'use_group_cv': config.USE_GROUP_CV,
                    'dice_weight': config.DICE_WEIGHT,
                    'ce_weight': config.BCE_WEIGHT
                }
            }, model_path)
            
            print(f"New best model saved! Dice: {best_dice:.6f}")
        else:
            patience_counter += 1
            print(f"No improvement. Patience count: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
            
            if patience_counter >= config.EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break
        
        # Memory cleanup
        torch.cuda.empty_cache()
    
    # Record results for current fold
    fold_result = {
        'fold': fold_idx,
        'best_dice': best_dice,
        'best_epoch': best_epoch,
        'final_val_loss': val_loss,
        'final_val_dice': val_dice,
        'final_val_iou': val_iou,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_dice_scores': val_dice_scores,
        'val_iou_scores': val_iou_scores,
        'val_dice_per_class': val_dice_per_class,
        'val_iou_per_class': val_iou_per_class
    }
    all_fold_results.append(fold_result)
    
    print(f"\nFold {fold_idx} training completed!")
    print(f"Best Dice Score: {best_dice:.6f} (Epoch {best_epoch})")
    print(f"Final Val Dice: {val_dice:.6f}")
    print(f"Final Val IoU: {val_iou:.6f}")
    
    # Clean up model and dataset for current fold
    del model, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*70}")
print("All 5 folds trained successfully!")
print(f"{'='*70}")

# Calculate and display statistical results for all folds
all_best_dices = [result['best_dice'] for result in all_fold_results]
all_final_dices = [result['final_val_dice'] for result in all_fold_results]
all_final_ious = [result['final_val_iou'] for result in all_fold_results]

print(f"\n=== 5-Fold Cross Validation Results Summary ===")
print(f"Best Dice Scores:")
for i, dice in enumerate(all_best_dices):
    print(f"  Fold {i}: {dice:.6f}")

print(f"\nFinal Validation Dice Scores:")
for i, dice in enumerate(all_final_dices):
    print(f"  Fold {i}: {dice:.6f}")

print(f"\nFinal Validation IoU Scores:")
for i, iou in enumerate(all_final_ious):
    print(f"  Fold {i}: {iou:.6f}")

print(f"\n=== Statistical Summary ===")
print(f"Best Dice Score - Mean: {np.mean(all_best_dices):.6f}, Std: {np.std(all_best_dices):.6f}")
print(f"Final Val Dice - Mean: {np.mean(all_final_dices):.6f}, Std: {np.std(all_final_dices):.6f}")
print(f"Final Val IoU - Mean: {np.mean(all_final_ious):.6f}, Std: {np.std(all_final_ious):.6f}")

print(f"\n=== Model File Save Locations ===")
for i in range(config.NUM_FOLDS):
    model_path = os.path.join(config.OUTPUT_DIR, f"{config.MODEL_NAME}_fold{i}_best.pth")
    if os.path.exists(model_path):
        file_size = os.path.getsize(model_path) / (1024*1024)
        print(f"  Fold {i}: {model_path} ({file_size:.1f} MB)")

print(f"\nAll 5 fold models have been saved and are ready for ensemble inference!")





=== Starting 5-Fold Cross Validation Training ===
Total folds: 5
Will sequentially train all 5 folds

Starting to train Fold 1/5
Fold 0 data split:
  Training set size: 3478
  Validation set size: 870
  Training set aneurysm positive ratio: 0.422
  Validation set aneurysm positive ratio: 0.455
  Training set modality distribution: {'CTA': 1459, 'MRA': 988, 'MRI T2': 790, 'MRI T1post': 241}
  Validation set modality distribution: {'CTA': 349, 'MRA': 264, 'MRI T2': 193, 'MRI T1post': 64}

Creating dataset for Fold 0...
Slice filtering statistics: Total slices 14364, Retained slices 14364, Filter rate 0.0%
Dataset initialization complete: 14364 valid samples (Minimum foreground pixel ratio: 0.0)
Slice filtering statistics: Total slices 3186, Retained slices 3186, Filter rate 0.0%
Dataset initialization complete: 3186 valid samples (Minimum foreground pixel ratio: 0.0)
Fold 0 data loaders:
  Training batches: 897
  Validation batches: 200

Initializing model for Fold 0...

Starting trainin

In [2]:
# === Inference pipeline for 5-fold UNet segmentation (2D) ===
import os, glob, sys, time, math
from typing import List, Tuple, Optional, Dict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import pydicom
import nibabel as nib
import SimpleITK as sitk
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# 0) Config (edit to your paths)
# -------------------------
class Config:
    # paths
    DICOM_SERIES_DIR = "/kaggle/input/rsna-intracranial-aneurysm-detection/kaggle_evaluation/series"  # folder containing subfolders named by SeriesInstanceUID
    SEGMENTATION_DIR = ""  # where seg/cowseg .nii live (optional)
    MODEL_DIR = "/kaggle/input/model1029/pytorch/default/1"  # folder containing your 5 .pth files
    MODEL_PATTERNS = ["2d_unet_segmentation_fold0_best.pth","2d_unet_segmentation_fold1_best.pth","2d_unet_segmentation_fold2_best.pth","2d_unet_segmentation_fold3_best.pth","2d_unet_segmentation_fold4_best.pth"]  # adjust if names differ
    OUT_DIR = "/kaggle/working/inference_out"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    IMAGE_SIZE = 224
    NUM_CLASSES = 14
    BATCH_SIZE = 8  # per-slice batch size during inference
    USE_CLAHE = False
    DEFAULT_WINDOW = (40.0, 80.0)  # fallback center,width
    SAVE_AS_NII = True

cfg = Config()
os.makedirs(cfg.OUT_DIR, exist_ok=True)

# -------------------------
# 1) Model definition (match your trained model)
# -------------------------
class UNet2D(nn.Module):
    def __init__(self, in_channels=1, num_classes=14, base_features=64):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, base_features)
        self.enc2 = self._conv_block(base_features, base_features*2)
        self.enc3 = self._conv_block(base_features*2, base_features*4)
        self.enc4 = self._conv_block(base_features*4, base_features*8)
        self.bottleneck = self._conv_block(base_features*8, base_features*16)
        self.upconv4 = nn.ConvTranspose2d(base_features*16, base_features*8, 2, 2)
        self.dec4 = self._conv_block(base_features*16, base_features*8)
        self.upconv3 = nn.ConvTranspose2d(base_features*8, base_features*4, 2, 2)
        self.dec3 = self._conv_block(base_features*8, base_features*4)
        self.upconv2 = nn.ConvTranspose2d(base_features*4, base_features*2, 2, 2)
        self.dec2 = self._conv_block(base_features*4, base_features*2)
        self.upconv1 = nn.ConvTranspose2d(base_features*2, base_features, 2, 2)
        self.dec1 = self._conv_block(base_features*2, base_features)
        self.final_conv = nn.Conv2d(base_features, num_classes, kernel_size=1)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1,2))
        e3 = self.enc3(F.max_pool2d(e2,2))
        e4 = self.enc4(F.max_pool2d(e3,2))
        b  = self.bottleneck(F.max_pool2d(e4,2))
        d4 = self.upconv4(b); d4 = torch.cat([d4, e4], dim=1); d4 = self.dec4(d4)
        d3 = self.upconv3(d4); d3 = torch.cat([d3, e3], dim=1); d3 = self.dec3(d3)
        d2 = self.upconv2(d3); d2 = torch.cat([d2, e2], dim=1); d2 = self.dec2(d2)
        d1 = self.upconv1(d2); d1 = torch.cat([d1, e1], dim=1); d1 = self.dec1(d1)
        out = self.final_conv(d1)
        return out

# -------------------------
# 2) Robust checkpoint loading helpers
# -------------------------
import inspect
def robust_torch_load(path, map_location):
    """
    Try multiple torch.load styles to handle different checkpoint formats.
    Returns whatever torch.load returns (state_dict or full dict).
    """
    # simple attempt first
    try:
        ck = torch.load(path, map_location=map_location)
        return ck
    except Exception as e:
        print("torch.load simple failed:", e)
    # try weights_only kw arg if available
    try:
        sig = inspect.signature(torch.load)
        if 'weights_only' in sig.parameters:
            try:
                ck = torch.load(path, map_location=map_location, weights_only=True)
                return ck
            except Exception as e2:
                print("torch.load(weights_only=True) failed:", e2)
            try:
                # last resort (unsafe) if user trusts the file
                print("Trying torch.load(..., weights_only=False) - only do this if checkpoint is trusted")
                ck = torch.load(path, map_location=map_location, weights_only=False)
                return ck
            except Exception as e3:
                print("torch.load(weights_only=False) failed:", e3)
    except Exception:
        pass
    # give up
    raise RuntimeError(f"Failed to load checkpoint: {path}")

def extract_state_dict(maybe_ck):
    """
    Accepts:
      - a state_dict (mapping)
      - a dict with keys 'model_state_dict' or 'state_dict'
      - a dict saved customly
    Returns a state_dict mapping or raises.
    """
    if isinstance(maybe_ck, dict):
        if 'model_state_dict' in maybe_ck:
            return maybe_ck['model_state_dict']
        if 'state_dict' in maybe_ck:
            return maybe_ck['state_dict']
        # maybe they saved only the state_dict already (keys are tensors)
        # Heuristic: if values look like tensors, treat as state_dict
        sample_vals = list(maybe_ck.values())[:5]
        if all(hasattr(v, 'dim') or isinstance(v, torch.Tensor) for v in sample_vals):
            return maybe_ck
    # Could be direct state_dict
    if isinstance(maybe_ck, (dict, )):
        return maybe_ck
    raise RuntimeError("Can't extract state_dict from checkpoint object")

def load_model_from_path(path, device, num_classes=14):
    print("Loading model:", path)
    ck = robust_torch_load(path, map_location=device)
    try:
        sd = extract_state_dict(ck)
    except Exception as e:
        print("Error extracting state dict:", e)
        raise
    model = UNet2D(in_channels=1, num_classes=num_classes)
    model.load_state_dict(sd)
    model.to(device)
    model.eval()
    return model

# -------------------------
# 3) Find the 5 model files
# -------------------------
def find_fold_paths(model_dir, patterns):
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(model_dir, pat)))
    files = sorted(files)
    if len(files) == 0:
        print("No model files found with patterns. Listing directory contents for debugging:", os.listdir(model_dir))
    return files

fold_paths = find_fold_paths(cfg.MODEL_DIR, cfg.MODEL_PATTERNS)
print("Found model files:", fold_paths)

# -------------------------
# 4) DICOM series loading & preprocessing helpers
# -------------------------
def load_dicom_volume(series_uid: str, dicom_root: str) -> Tuple[np.ndarray, List[str], dict]:
    """
    Load a DICOM series (folder named by series_uid) into a numpy volume (Z,H,W).
    Returns: (vol, sorted_filenames, sample_metadata_from_first_slice)
    """
    folder = os.path.join(dicom_root, series_uid)
    if not os.path.isdir(folder):
        raise FileNotFoundError(folder)
    # get dcm files
    dcm_files = [os.path.join(folder,f) for f in os.listdir(folder) if f.lower().endswith('.dcm')]
    if not dcm_files:
        raise FileNotFoundError("No DICOMs in "+folder)
    # sort by InstanceNumber when available, otherwise filename
    def instnum(p):
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True, force=True)
            return int(getattr(ds, 'InstanceNumber', 0))
        except Exception:
            return 0
    dcm_files = sorted(dcm_files, key=instnum)
    imgs = []
    meta = {}
    for i,p in enumerate(dcm_files):
        ds = pydicom.dcmread(p, stop_before_pixels=False, force=True)
        arr = ds.pixel_array
        # if arr is multichannel or 3D skip (we expect 2D slice)
        if arr.ndim != 2:
            # try to handle some weird shapes by selecting first channel
            if arr.ndim == 3:
                arr = arr[..., 0]
            else:
                raise RuntimeError(f"Unexpected shape in {p}: {arr.shape}")
        imgs.append(arr.astype(np.float32))
        if i == 0:
            meta['WindowCenter'] = getattr(ds, 'WindowCenter', None)
            meta['WindowWidth']  = getattr(ds, 'WindowWidth', None)
            meta['Modality'] = getattr(ds, 'Modality', None)
    vol = np.stack(imgs, axis=0)  # Z,H,W
    return vol, dcm_files, meta

def get_window_from_meta(meta):
    wc = meta.get('WindowCenter')
    ww = meta.get('WindowWidth')
    # WindowCenter/Width can be sequences -> pick first
    if isinstance(wc, (list, tuple, pydicom.multival.MultiValue)):
        wc = wc[0]
    if isinstance(ww, (list, tuple, pydicom.multival.MultiValue)):
        ww = ww[0]
    try:
        wc = float(wc); ww = float(ww)
        return (wc, ww)
    except Exception:
        return cfg.DEFAULT_WINDOW

def apply_window(img2d, wc, ww):
    """Clip by window center/width and scale 0-255 (uint8)"""
    img_min = wc - ww/2.0
    img_max = wc + ww/2.0
    img = np.clip(img, img_min, img_max)
    img = (img - img_min) / (img_max - img_min + 1e-7)
    img = (img * 255.0).astype(np.uint8)
    return img

def robust_normalize_uint8(img2d):
    """Fallback normalization to 0..255 using percentile clipping"""
    lo, hi = np.percentile(img2d, [1, 99])
    img = np.clip(img2d, lo, hi)
    img = ((img - lo) / (hi - lo + 1e-7) * 255.0).astype(np.uint8)
    return img

def preprocess_slice_for_model(slice2d: np.ndarray, wc_ww: Tuple[float,float], target_size:int):
    """Return float32 tensor in range 0..1 sized (1,H,W)"""
    try:
        img = apply_window(slice2d, wc_ww[0], wc_ww[1])
    except Exception:
        img = robust_normalize_uint8(slice2d)
    # optionally CLAHE
    if cfg.USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = clahe.apply(img)
    # resize
    if img.shape != (target_size,target_size):
        img = cv2.resize(img, (target_size,target_size), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)  # 1,H,W
    return img

# -------------------------
# 5) Optional: load NIfTI segmentation aligned to dicom series
# -------------------------
def load_nii_and_resample_to_dicom(nii_path: str, dicom_vol_sitk: sitk.Image) -> np.ndarray:
    """Load a NIfTI file and resample (nearest neighbor) to match dicom_vol_sitk"""
    seg = sitk.ReadImage(nii_path)
    # convert seg to same spacing/origin/orientation as dicom
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(dicom_vol_sitk)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampled = resampler.Execute(seg)
    arr = sitk.GetArrayFromImage(resampled)  # z,y,x (Z,H,W)
    return arr

# -------------------------
# 6) Inference helpers (ensemble)
# -------------------------
def load_models_fold_list(paths: List[str], device, num_classes=14):
    models = []
    for p in paths:
        try:
            m = load_model_from_path(p, device, num_classes=num_classes)
            models.append(m)
        except Exception as e:
            print("Failed to load model", p, ":", e)
    if not models:
        raise RuntimeError("No models loaded")
    return models

def inference_on_series(series_uid: str, models: List[nn.Module], dicom_root: str, batch_size:int=8):
    """
    Loads DICOM series, preprocess per slice, run ensemble inference and return predicted volume (Z,H,W ints)
    """
    vol, file_list, meta = load_dicom_volume(series_uid, dicom_root)
    wc_ww = get_window_from_meta(meta)
    Z,H,W = vol.shape
    preds_prob = np.zeros((len(models), cfg.NUM_CLASSES, Z, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE), dtype=np.float32)
    # For memory: process per-slice batches
    slices = [preprocess_slice_for_model(vol[z], wc_ww, cfg.IMAGE_SIZE) for z in range(Z)]
    slices = np.stack(slices, axis=0)  # Z,1,H,W
    # run by folds
    for m_i, m in enumerate(models):
        all_out = []
        m.to(cfg.DEVICE)
        m.eval()
        with torch.no_grad():
            for i in range(0, Z, batch_size):
                batch = torch.from_numpy(slices[i:i+batch_size]).to(cfg.DEVICE).float()  # B,1,H,W
                logits = m(batch)  # B, C, H, W
                probs = torch.softmax(logits, dim=1).cpu().numpy()  # B,C,H,W
                all_out.append(probs)
        probs_all = np.concatenate(all_out, axis=0)  # Z,C,H,W
        # reorder to C,Z,H,W
        preds_prob[m_i] = probs_all.transpose(1,0,2,3)
    # average across folds
    avg_prob = preds_prob.mean(axis=0)  # C,Z,H,W
    # convert to predicted label per voxel per-slice
    pred_labels = np.argmax(avg_prob, axis=0)  # Z,H,W
    # also produce per-class softmax volumes if needed
    return pred_labels, avg_prob  # pred_labels: Z,H,W  ; avg_prob: C,Z,H,W

# -------------------------
# 7) Ensemble load and run over a list of series UIDs
# -------------------------
def run_inference_for_series_list(series_uids: List[str], model_paths: List[str], out_dir: str):
    models = load_models_fold_list(model_paths, cfg.DEVICE, num_classes=cfg.NUM_CLASSES)
    results = {}
    for i, sid in enumerate(series_uids):
        t0 = time.time()
        try:
            pred_labels, avg_prob = inference_on_series(sid, models, cfg.DICOM_SERIES_DIR, batch_size=cfg.BATCH_SIZE)
            # save predicted volume as npy
            np.save(os.path.join(out_dir, f"{sid}_pred_labels.npy"), pred_labels.astype(np.uint8))
            # optional save each class probability maps (space heavy)
            # np.save(os.path.join(out_dir, f"{sid}_pred_probs.npy"), avg_prob.astype(np.float32))
            # optionally save as NIfTI (need affine). If a seg .nii exists, use its affine, else create identity affine
            seg_nifti_candidate = os.path.join(cfg.SEGMENTATION_DIR, f"{sid}.nii")
            if not os.path.exists(seg_nifti_candidate):
                seg_nifti_candidate = os.path.join(cfg.SEGMENTATION_DIR, f"{sid}_cowseg.nii")
            if cfg.SAVE_AS_NII and os.path.exists(seg_nifti_candidate):
                ref = nib.load(seg_nifti_candidate)
                affine = ref.affine
                out_img = nib.Nifti1Image(pred_labels.astype(np.uint8), affine)
                nib.save(out_img, os.path.join(out_dir, f"{sid}_pred_labels.nii.gz"))
            elif cfg.SAVE_AS_NII:
                # create simple affine (voxel-based)
                affine = np.eye(4)
                out_img = nib.Nifti1Image(pred_labels.astype(np.uint8), affine)
                nib.save(out_img, os.path.join(out_dir, f"{sid}_pred_labels.nii.gz"))
            # compute per-series "aneurysm present" binary label
            aneurysm_present = int((pred_labels > 0).any())
            results[sid] = {
                "aneurysm_present": aneurysm_present,
                "pred_npy": os.path.join(out_dir, f"{sid}_pred_labels.npy"),
                "pred_nii": os.path.join(out_dir, f"{sid}_pred_labels.nii.gz")
            }
            dt = time.time() - t0
            print(f"[{i+1}/{len(series_uids)}] {sid} -> done in {dt:.1f}s  aneurysm_present={aneurysm_present}")
        except Exception as e:
            print(f"[{i+1}/{len(series_uids)}] {sid} -> FAILED:", e)
            results[sid] = {"error": str(e)}
    return results

# -------------------------
# 8) Optional: compute Dice per class (if seg.nii or cowseg available)
# -------------------------
def compute_dice_from_volumes(pred_vol: np.ndarray, target_vol: np.ndarray, num_classes: int):
    """
    pred_vol, target_vol: shape (Z,H,W) integer labels
    returns dice_per_class list length num_classes
    """
    dices = []
    smooth = 1e-6
    for c in range(num_classes):
        p = (pred_vol == c).astype(np.float32)
        t = (target_vol == c).astype(np.float32)
        inter = (p * t).sum()
        dice = (2.0 * inter + smooth) / (p.sum() + t.sum() + smooth)
        dices.append(float(dice))
    return dices

def load_seg_nifti_for_series(series_uid):
    # try cowseg then seg
    cand = os.path.join(cfg.SEGMENTATION_DIR, f"{series_uid}_cowseg.nii")
    if not os.path.exists(cand):
        cand = os.path.join(cfg.SEGMENTATION_DIR, f"{series_uid}.nii")
    if not os.path.exists(cand):
        return None
    # attempt resampling to dicom grid using SimpleITK
    # get dicom ref
    folder = os.path.join(cfg.DICOM_SERIES_DIR, series_uid)
    try:
        reader = sitk.ImageSeriesReader()
        files = reader.GetGDCMSeriesFileNames(folder)
        if len(files) == 0:
            return None
        reader.SetFileNames(files)
        dicom_img = reader.Execute()
        target_arr = load_nii_and_resample_to_dicom(cand, dicom_img)  # Z,H,W
        return target_arr.astype(np.uint8)
    except Exception:
        # fallback: load raw nii and transpose if needed
        raw = nib.load(cand).get_fdata()
        # many niftis are (W,H,Z) or (X,Y,Z) -> attempt to bring to (Z,H,W)
        if raw.ndim == 3:
            # try common shapes
            if raw.shape[2] == len(os.listdir(folder)):
                vol = np.transpose(raw, (2,1,0)) if raw.shape[2] != len(os.listdir(folder)) else raw
            else:
                vol = raw
        else:
            vol = raw
        return np.asarray(vol).astype(np.uint8)

# -------------------------
# 9) Example usage
# -------------------------
if __name__ == "__main__":
    # 1) pick a list of series to run (here all folders in DICOM_SERIES_DIR)
    series_folders = [f for f in os.listdir(cfg.DICOM_SERIES_DIR) if os.path.isdir(os.path.join(cfg.DICOM_SERIES_DIR, f))]
    # for quick test you might pick first N
    series_to_run = series_folders[:20]  # change to full list when ready
    print("Will run on", len(series_to_run), "series")

    # 2) ensure model files exist
    if len(fold_paths) < 1:
        print("No model files found - stopping")
    else:
        # 3) run inference (this loads models once)
        results = run_inference_for_series_list(series_to_run, fold_paths, cfg.OUT_DIR)

        # 4) optional: if seg masks available, compute dice
        dices_summary = {}
        for sid, info in results.items():
            if info.get("error"): continue
            pred_npy = info['pred_npy']
            pred_vol = np.load(pred_npy)
            gt = load_seg_nifti_for_series(sid)
            if gt is None:
                continue
            # if shapes mismatch, you may need to resample; we assume they match
            if pred_vol.shape != gt.shape:
                print("Shape mismatch (pred,gt)", pred_vol.shape, gt.shape, "for", sid)
                continue
            dices = compute_dice_from_volumes(pred_vol, gt, cfg.NUM_CLASSES)
            dices_summary[sid] = dices
            print(sid, "avg_dice", np.mean(dices), "per-class (first 5):", dices[:5])

        # Save a CSV summary
        import pandas as pd
        rows = []
        for sid, info in results.items():
            row = {"SeriesInstanceUID": sid}
            row.update(info)
            if sid in dices_summary:
                row['avg_dice'] = float(np.mean(dices_summary[sid]))
            rows.append(row)
        pd.DataFrame(rows).to_csv(os.path.join(cfg.OUT_DIR, "inference_results_summary.csv"), index=False)
        print("Inference finished. Results saved to", cfg.OUT_DIR)


Found model files: ['/kaggle/input/model1029/pytorch/default/1/2d_unet_segmentation_fold0_best.pth', '/kaggle/input/model1029/pytorch/default/1/2d_unet_segmentation_fold1_best.pth', '/kaggle/input/model1029/pytorch/default/1/2d_unet_segmentation_fold2_best.pth', '/kaggle/input/model1029/pytorch/default/1/2d_unet_segmentation_fold3_best.pth', '/kaggle/input/model1029/pytorch/default/1/2d_unet_segmentation_fold4_best.pth']
Will run on 3 series
Loading model: /kaggle/input/model1029/pytorch/default/1/2d_unet_segmentation_fold0_best.pth
torch.load simple failed: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from